In [ ]:
# ============================================================
# Cell 1 — Depression (MDD) Algorithm: full notebook-ready code
#        Multi-path routing per Step 2 of CLAUDE_MDD.md
# ============================================================
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any, Callable, Dict, List, Literal, Optional, Set, Tuple

from pathlib import Path
import re

from pydantic import BaseModel, Field, field_validator


# ------------------------------------------------------------
# 1) Medication Knowledge Base (KB)
# ------------------------------------------------------------
MedicationKey = str

MED_KB: Dict[MedicationKey, Dict[str, Any]] = {
    # --- SSRIs ---
    "fluoxetine": {
        "display": "Fluoxetine (Prozac)",
        "class": "SSRI",
        "start_dose": "10–20 mg daily",
        "dose_range": "20–80 mg daily",
        "titration": "Increase by 10–20 mg every 1–2 weeks as tolerated; reassess at 6 weeks.",
        "monitoring": [],
        "bp_message_relevant": False,
        "qt_risk": "none",
    },
    "sertraline": {
        "display": "Sertraline (Zoloft)",
        "class": "SSRI",
        "start_dose": "25–50 mg daily",
        "dose_range": "50–200 mg daily",
        "titration": "Increase by 25–50 mg every 1–2 weeks as tolerated; reassess at 6 weeks.",
        "monitoring": [],
        "bp_message_relevant": False,
        "qt_risk": "none",
    },
    "escitalopram": {
        "display": "Escitalopram (Lexapro)",
        "class": "SSRI",
        "start_dose": "5–10 mg daily",
        "dose_range": "10–20 mg daily",
        "titration": "Increase by 5–10 mg every 1–2 weeks as tolerated; reassess at 6 weeks.",
        "monitoring": [],
        "bp_message_relevant": False,
        "qt_risk": "possible",
    },
    "citalopram": {
        "display": "Citalopram (Celexa)",
        "class": "SSRI",
        "start_dose": "10–20 mg daily",
        "dose_range": "20–40 mg daily",
        "titration": "Increase by 10–20 mg every 1–2 weeks as tolerated; reassess at 6 weeks.",
        "monitoring": [],
        "bp_message_relevant": False,
        "qt_risk": "known",
    },
    "paroxetine": {
        "display": "Paroxetine (Paxil)",
        "class": "SSRI",
        "start_dose": "10–20 mg daily",
        "dose_range": "20–50 mg daily",
        "titration": "Increase by 10 mg every 1–2 weeks as tolerated; reassess at 6 weeks.",
        "monitoring": [],
        "bp_message_relevant": False,
        "qt_risk": "none",
    },
    "fluvoxamine": {
        "display": "Fluvoxamine (Luvox) [off-label for MDD — 25, 24]",
        "class": "SSRI",
        "start_dose": "50 mg daily",
        "dose_range": "100–300 mg daily",
        "titration": "Increase by 50 mg every 1–2 weeks as tolerated; reassess at 6 weeks.",
        "monitoring": [],
        "bp_message_relevant": False,
        "qt_risk": "none",
    },
    # --- SNRIs ---
    "venlafaxine_xr": {
        "display": "Venlafaxine XR (Effexor XR)",
        "class": "SNRI",
        "start_dose": "37.5–75 mg daily",
        "dose_range": "75–225 mg daily",
        "titration": "Increase by 37.5–75 mg every 1–2 weeks as tolerated; reassess at 6 weeks.",
        "monitoring": ["BP check before prescribing; monitor BP during titration."],
        "bp_message_relevant": True,
        "qt_risk": "none",
    },
    "duloxetine": {
        "display": "Duloxetine (Cymbalta)",
        "class": "SNRI",
        "start_dose": "30–40 mg daily",
        "dose_range": "60 mg daily",
        "titration": "Increase to 60 mg after 1–2 weeks if tolerated; reassess at 6 weeks. Max 60 mg/day [22].",
        "monitoring": ["BP check before prescribing; monitor BP during titration."],
        "bp_message_relevant": True,
        "qt_risk": "none",
    },
    "desvenlafaxine": {
        "display": "Desvenlafaxine (Pristiq)",
        "class": "SNRI",
        "start_dose": "50 mg daily",
        "dose_range": "50–100 mg daily",
        "titration": "Increase by 50 mg as needed; reassess at 6 weeks.",
        "monitoring": ["BP check before prescribing; monitor BP during titration."],
        "bp_message_relevant": True,
        "qt_risk": "none",
    },
    "levomilnacipran_er": {
        "display": "Levomilnacipran ER (Fetzima)",
        "class": "SNRI",
        "start_dose": "20 mg daily",
        "dose_range": "40–120 mg daily",
        "titration": "Increase by 20–40 mg every 1–2 weeks as tolerated; reassess at 6 weeks.",
        "monitoring": ["BP check before prescribing; monitor BP during titration."],
        "bp_message_relevant": True,
        "qt_risk": "none",
    },
    # --- Other antidepressants ---
    "bupropion_xl": {
        "display": "Bupropion XL (Wellbutrin XL)",
        "class": "NDRI",
        "start_dose": "150 mg daily",
        "dose_range": "150–450 mg daily",
        "titration": "Increase to 300 mg daily after 3–7 days if tolerated; max 450 mg daily.",
        "monitoring": ["BP check before prescribing; monitor BP during titration."],
        "bp_message_relevant": True,
        "qt_risk": "none",
    },
    "mirtazapine": {
        "display": "Mirtazapine (Remeron)",
        "class": "NaSSA",
        "start_dose": "15 mg nightly",
        "dose_range": "15–45 mg nightly",
        "titration": "Increase by 15 mg every 1–2 weeks as tolerated; reassess at 6 weeks.",
        "monitoring": [],
        "bp_message_relevant": False,
        "qt_risk": "none",
    },
    "vortioxetine": {
        "display": "Vortioxetine (Trintellix)",
        "class": "Other",
        "start_dose": "10 mg daily",
        "dose_range": "10–20 mg daily",
        "titration": "Increase to 20 mg after 1–2 weeks if tolerated; reassess at 6 weeks.",
        "monitoring": [],
        "bp_message_relevant": False,
        "qt_risk": "none",
    },
    "vilazodone": {
        "display": "Vilazodone (Viibryd)",
        "class": "Other",
        "start_dose": "10 mg daily",
        "dose_range": "20–40 mg daily",
        "titration": "Increase to 20 mg after 1 week, then 40 mg after another week if tolerated.",
        "monitoring": [],
        "bp_message_relevant": False,
        "qt_risk": "none",
    },
    # --- Augmentation ---
    "aripiprazole": {
        "display": "Aripiprazole (Abilify) — augmentation",
        "class": "SGA",
        "start_dose": "2–5 mg daily",
        "dose_range": "2–15 mg daily",
        "titration": "Increase by 2–5 mg every 1–2 weeks as needed/tolerated; reassess at 6 weeks.",
        "monitoring": ["Metabolic monitoring (weight/BMI, BP, lipids, glucose/A1c) per local protocol."],
        "bp_message_relevant": False,
        "qt_risk": "none",
    },
    "lithium": {
        "display": "Lithium — augmentation",
        "class": "Augment",
        "start_dose": "Clinician-managed",
        "dose_range": "Target 0.6–1.2 mEq/L (0.3–0.6 mmol/L in older adults) [54]",
        "titration": "Requires individualized titration with serum level monitoring.",
        "monitoring": ["Baseline and ongoing: Li+ level, TSH, BMP. Check at initiation, 1–2 months, then every 6–12 months [54]."],
        "bp_message_relevant": False,
        "qt_risk": "none",
    },
    "brexpiprazole": {
        "display": "Brexpiprazole (Rexulti) — augmentation",
        "class": "SGA",
        "start_dose": "0.5–1 mg daily",
        "dose_range": "1–3 mg daily",
        "titration": "Titrate weekly as tolerated; reassess at 6 weeks [44].",
        "monitoring": ["Metabolic monitoring (weight/BMI, BP, lipids, glucose/A1c) at initiation, 3 months, then annually."],
        "bp_message_relevant": False,
        "qt_risk": "none",
    },
    "quetiapine": {
        "display": "Quetiapine XR (Seroquel XR) — augmentation",
        "class": "SGA",
        "start_dose": "50 mg nightly",
        "dose_range": "150–300 mg/day",
        "titration": "Titrate every 1–2 weeks toward 150–300 mg/day; sedation common at initiation [44, 54].",
        "monitoring": ["Metabolic monitoring (weight/BMI, BP, lipids, glucose/A1c) at initiation, 3 months, then annually."],
        "bp_message_relevant": False,
        "qt_risk": "moderate",
    },
    "cariprazine": {
        "display": "Cariprazine (Vraylar) — augmentation",
        "class": "SGA",
        "start_dose": "0.5 mg daily",
        "dose_range": "1.5–3 mg daily",
        "titration": "Titrate weekly; slow titration reduces akathisia risk [44].",
        "monitoring": ["Metabolic monitoring (weight/BMI, BP, lipids, glucose/A1c) at initiation, 3 months, then annually."],
        "bp_message_relevant": False,
        "qt_risk": "none",
    },
    "risperidone": {
        "display": "Risperidone — augmentation",
        "class": "SGA",
        "start_dose": "0.25–0.5 mg daily",
        "dose_range": "0.5–2 mg daily",
        "titration": "Titrate gradually; monitor for EPS and hyperprolactinemia [44].",
        "monitoring": ["Metabolic monitoring; monitor for EPS, hyperprolactinemia."],
        "bp_message_relevant": False,
        "qt_risk": "low",
    },
    "olanzapine": {
        "display": "Olanzapine — augmentation",
        "class": "SGA",
        "start_dose": "2.5–5 mg daily",
        "dose_range": "5–20 mg daily",
        "titration": "Titrate as tolerated; olanzapine/fluoxetine combination pill (Symbyax) available [44].",
        "monitoring": ["Metabolic monitoring required — significant weight gain and metabolic risk."],
        "bp_message_relevant": False,
        "qt_risk": "none",
    },
    "methylphenidate": {
        "display": "Methylphenidate — augmentation (residual anergia only)",
        "class": "Stimulant",
        "start_dose": "5–10 mg twice daily",
        "dose_range": "10–60 mg/day",
        "titration": "Titrate weekly; limited evidence in MDD augmentation; use only for prominent residual anergia [3, 52].",
        "monitoring": ["Monitor HR, BP; assess for abuse potential."],
        "bp_message_relevant": True,
        "qt_risk": "none",
    },
    "lurasidone": {
        "display": "Lurasidone (Latuda) — augmentation",
        "class": "SGA",
        "start_dose": "20–40 mg/day",
        "dose_range": "20–80 mg/day",
        "titration": "Titrate as tolerated; limited evidence in unipolar depression; most data in bipolar — use cautiously [43].",
        "monitoring": ["Metabolic monitoring (weight/BMI, BP, lipids, glucose/A1c) at initiation, 3 months, then annually."],
        "bp_message_relevant": False,
        "qt_risk": "none",
    },
    "lisdexamfetamine": {
        "display": "Lisdexamfetamine (Vyvanse) — augmentation [OFF-LABEL for MDD]",
        "class": "Stimulant",
        "start_dose": "20–30 mg daily",
        "dose_range": "20–70 mg/day",
        "titration": "Titrate weekly; FDA-approved for binge eating disorder only — off-label for MDD. One positive RCT for augmentation [43, 59].",
        "monitoring": ["Monitor HR, BP; assess for abuse potential; document off-label use."],
        "bp_message_relevant": True,
        "qt_risk": "none",
    },
}

DOSE_STEPS_MG: Dict[str, List[float]] = {
    "sertraline":       [25, 50, 100, 150, 200],
    "escitalopram":     [5, 10, 20],
    "fluoxetine":       [10, 20, 40, 60, 80],
    "citalopram":       [10, 20, 40],
    "paroxetine":       [10, 20, 30, 40, 50],
    "fluvoxamine":      [50, 100, 150, 200, 300],
    "venlafaxine_xr":   [37.5, 75, 150, 225],
    "duloxetine":       [30, 60],
    "desvenlafaxine":   [50, 100],
    "levomilnacipran_er": [20, 40, 80, 120],
    "bupropion_xl":     [150, 300, 450],
    "mirtazapine":      [15, 30, 45],
    "vortioxetine":     [10, 20],
    "vilazodone":       [10, 20, 40],
    "aripiprazole":     [2, 5, 10, 15],
    "brexpiprazole":    [0.5, 1, 2, 3],
    "quetiapine":       [50, 100, 150, 200, 300],
    "cariprazine":      [0.5, 1.5, 3],
    "risperidone":      [0.25, 0.5, 1, 2],
    "olanzapine":       [2.5, 5, 10, 15, 20],
    "methylphenidate":  [5, 10, 20, 30, 40, 60],
    "lurasidone":       [20, 40, 60, 80],
    "lisdexamfetamine": [20, 30, 40, 50, 60, 70],
}

DOSE_MIN_MAX_MG: Dict[str, Tuple[float, float]] = {
    "sertraline":         (25, 200),
    "escitalopram":       (5, 20),
    "fluoxetine":         (10, 80),
    "citalopram":         (10, 40),
    "paroxetine":         (10, 50),
    "fluvoxamine":        (50, 300),
    "venlafaxine_xr":     (37.5, 225),
    "duloxetine":         (30, 60),
    "desvenlafaxine":     (50, 100),
    "levomilnacipran_er": (20, 120),
    "bupropion_xl":       (150, 450),
    "mirtazapine":        (15, 45),
    "vortioxetine":       (10, 20),
    "vilazodone":         (10, 40),
    "aripiprazole":       (2, 15),
    "brexpiprazole":      (0.5, 3),
    "quetiapine":         (50, 300),
    "cariprazine":        (0.5, 3),
    "risperidone":        (0.25, 2),
    "olanzapine":         (2.5, 20),
    "methylphenidate":    (5, 60),
    "lurasidone":         (20, 80),
    "lisdexamfetamine":   (20, 70),
}

# Renal dose caps (CrCl/eGFR bucket-based)
RENAL_MAX_CAPS_MG: Dict[str, Dict[str, float]] = {
    "sertraline":     {">60": 200,   "30-60": 200,   "15-29": 150,   "<15": 100},
    "venlafaxine_xr": {">60": 225,   "30-60": 112.5, "15-29": 112.5, "<15": 75},
    "bupropion_xl":   {">60": 450,   "30-60": 300,   "15-29": 150,   "<15": 150},
}

def dose_out_of_range(med_key: str, dose_mg: float, max_cap_override: Optional[float] = None) -> bool:
    if med_key not in DOSE_MIN_MAX_MG:
        return False
    mn, mx = DOSE_MIN_MAX_MG[med_key]
    if max_cap_override is not None:
        mx = min(mx, float(max_cap_override))
    return dose_mg < mn or dose_mg > mx

def next_dose_step(med_key: str, current_dose_mg: float, cap_override: Optional[float] = None) -> str:
    if med_key in DOSE_MIN_MAX_MG:
        mn, mx = DOSE_MIN_MAX_MG[med_key]
        if cap_override is not None:
            mx = min(mx, float(cap_override))
        if current_dose_mg > mx:
            return f"Current dose ({current_dose_mg} mg) exceeds max cap ({mx} mg)—do not increase; clinician review."
        if current_dose_mg < mn:
            return f"Current dose ({current_dose_mg} mg) below typical min ({mn} mg)—verify dosing; clinician review."
    steps = DOSE_STEPS_MG.get(med_key, [])
    if not steps:
        return "Increase dose per standard titration schedule."
    if cap_override is not None:
        steps = [s for s in steps if s <= float(cap_override)]
        if not steps:
            return "Dose cap prevents further titration—clinician review."
    for s in steps:
        if s > current_dose_mg:
            return f"Increase dose to {s} mg daily (next step), if tolerated."
    return "Already at max dose (or max cap)—do not increase; consider augmentation or switch per algorithm."


# ------------------------------------------------------------
# 2) Medication safety screening
# ------------------------------------------------------------
def _norm(s: str) -> str:
    s = str(s or "").strip().lower()
    for ch in [",", ";", ":", "(", ")", "[", "]", "{", "}", "/", "\\", "|", '"', "'"]:
        s = s.replace(ch, " ")
    return " ".join(s.split())

def _any_match(patient_meds_norm: Set[str], synonyms: List[str]) -> bool:
    syn_norm = [_norm(x) for x in synonyms]
    for pm in patient_meds_norm:
        for sn in syn_norm:
            if sn and (pm == sn or sn in pm):
                return True
    return False

MAOI_SYNONYMS = [
    "phenelzine", "nardil", "tranylcypromine", "parnate", "isocarboxazid", "marplan",
    "selegiline", "eldepryl", "zelapar", "emsam", "selegiline patch", "selegiline transdermal",
    "rasagiline", "azilect", "linezolid", "zyvox", "methylene blue", "methylthioninium",
]
SEROTONERGIC_SYNONYMS = [
    "sumatriptan", "imitrex", "rizatriptan", "maxalt", "zolmitriptan", "zomig",
    "naratriptan", "amerge", "almotriptan", "axert", "eletriptan", "relpax", "frovatriptan", "frova",
    "tramadol", "ultram", "tapentadol", "nucynta", "meperidine", "demerol", "fentanyl", "methadone",
    "lithium", "lithobid", "eskalith", "buspirone", "buspar",
    "st johns wort", "st. john's wort", "hypericum", "same", "s-adenosylmethionine", "tryptophan",
]
BLEEDING_RISK_SYNONYMS = [
    "warfarin", "coumadin", "apixaban", "eliquis", "rivaroxaban", "xarelto",
    "dabigatran", "pradaxa", "edoxaban", "savaysa", "aspirin", "asa",
    "clopidogrel", "plavix", "ticagrelor", "brilinta", "prasugrel", "effient",
    "ibuprofen", "advil", "motrin", "naproxen", "aleve", "diclofenac", "voltaren", "ketorolac", "toradol",
]
QT_PROLONGING_SYNONYMS = [
    "ondansetron", "zofran", "droperidol", "amiodarone", "cordarone",
    "sotalol", "betapace", "dofetilide", "tikosyn", "quinidine", "procainamide",
    "azithromycin", "zithromax", "clarithromycin", "biaxin", "erythromycin",
    "levofloxacin", "levaquin", "moxifloxacin", "avelox",
    "ziprasidone", "geodon", "haloperidol", "haldol", "quetiapine", "seroquel",
    "chlorpromazine", "thorazine", "thioridazine", "mellaril", "pimozide", "orap",
    "hydroxychloroquine", "plaquenil", "methadone",
]
BP_RISK_SYNONYMS = [
    "amphetamine", "adderall", "dextroamphetamine", "dexedrine",
    "lisdexamfetamine", "vyvanse", "methylphenidate", "ritalin", "concerta",
    "midodrine", "proamatine", "fludrocortisone", "florinef",
    "pseudoephedrine", "sudafed", "phenylephrine",
]
SEIZURE_RISK_SYNONYMS = [
    "tramadol", "ultram", "theophylline", "clozapine", "clozaril",
    "prednisone", "methylprednisolone", "dexamethasone", "hydrocortisone",
    "amphetamine", "adderall", "dextroamphetamine", "dexedrine",
    "lisdexamfetamine", "vyvanse", "methylphenidate", "ritalin", "concerta",
]

@dataclass
class MedSafetyResult:
    blocked: Set[str]
    warnings: List[str]
    detected: Dict[str, List[str]]

def medication_safety_screen(patient_medications: List[str], candidate_meds: List[str]) -> MedSafetyResult:
    pm_norm = {_norm(m) for m in (patient_medications or []) if str(m).strip()}
    blocked: Set[str] = set()
    warnings: List[str] = []
    detected: Dict[str, List[str]] = {}

    has_maoi       = _any_match(pm_norm, MAOI_SYNONYMS)
    has_serotonergic = _any_match(pm_norm, SEROTONERGIC_SYNONYMS)
    has_bleed      = _any_match(pm_norm, BLEEDING_RISK_SYNONYMS)
    has_qt         = _any_match(pm_norm, QT_PROLONGING_SYNONYMS)
    has_bp_risk    = _any_match(pm_norm, BP_RISK_SYNONYMS)
    has_seizure    = _any_match(pm_norm, SEIZURE_RISK_SYNONYMS)

    if has_maoi:
        for cm in candidate_meds:
            if cm in MED_KB and MED_KB[cm]["class"] in {"SSRI", "SNRI", "Other", "NDRI", "NaSSA"}:
                blocked.add(cm)
        warnings.append("MAOI (or MAOI-like agent) detected: antidepressant initiation contraindicated. Route to clinician review.")
        detected["maoi_signal"] = ["present"]
    if has_serotonergic:
        warnings.append("Serotonergic co-med detected: monitor for serotonin syndrome; avoid stacking serotonergic agents.")
        detected["serotonergic_signal"] = ["present"]
    if has_bleed:
        warnings.append("Bleeding-risk medication detected (anticoagulant/antiplatelet/NSAID): SSRIs/SNRIs may increase bleeding risk.")
        detected["bleeding_signal"] = ["present"]
    if has_qt:
        warnings.append("QT-prolonging co-med detected: consider ECG review where clinically indicated.")
        detected["qt_signal"] = ["present"]
    if has_bp_risk:
        warnings.append("BP-elevating co-med detected: check and monitor BP during titration.")
        detected["bp_signal"] = ["present"]
    if has_seizure:
        warnings.append("Seizure-threshold-lowering factor detected: use caution with bupropion if considered.")
        detected["seizure_signal"] = ["present"]

    return MedSafetyResult(blocked=blocked, warnings=warnings, detected=detected)


# ------------------------------------------------------------
# 3) Data models
# ------------------------------------------------------------
ManiaScreen = Literal["positive", "negative", "unknown"]

# All recognised deviation path names (Step 2 of CLAUDE_MDD.md).
# A single patient may be on multiple paths simultaneously.
PathwayName = Literal[
    "peds", "antepartum", "postpartum",
    "renal", "hepatic", "cardiac",
    "geriatric", "obesity", "dementia",
    "pain", "insomnia",
    "adult",
]

RecommendationIntent = Literal[
    "start", "continue", "increase", "switch_to", "augment_with", "monitor_only",
]

class PriorTrial(BaseModel):
    medication_key: str
    outcome: Literal["remission_or_response", "partial", "no_response", "intolerant", "unknown"] = "unknown"

    @field_validator("medication_key")
    @classmethod
    def normalize_med_key(cls, v: str) -> str:
        return (v or "").strip().lower()

class PatientInput(BaseModel):
    # Core
    age: int = Field(..., ge=0, le=130)
    phq_current: int = Field(..., ge=0, le=27)
    mania_screen: ManiaScreen = "unknown"
    current_medications: List[str] = Field(default_factory=list)

    # Follow-up / episode
    baseline_phq: Optional[int] = Field(default=None, ge=0, le=27)
    weeks_on_current_antidepressant: Optional[int] = Field(default=None, ge=0, le=400)
    current_antidepressant_key: Optional[str] = None
    current_dose_mg: Optional[float] = Field(default=None, ge=0, le=2000)
    trial_number: Optional[int] = Field(default=None, ge=1, le=3)
    is_augmented: bool = False
    weeks_since_augmentation: Optional[int] = Field(default=None, ge=0, le=400)
    prior_trials: List[PriorTrial] = Field(default_factory=list)

    # Vitals
    bp_systolic: Optional[int] = Field(default=None, ge=50, le=260)
    bp_diastolic: Optional[int] = Field(default=None, ge=30, le=160)

    # Renal
    crcl_ml_min: Optional[float] = Field(default=None, ge=0, le=300)
    egfr_ml_min_1_73: Optional[float] = Field(default=None, ge=0, le=300)

    # Hepatic
    hepatic_impairment: bool = False
    child_pugh: Optional[Literal["A", "B", "C"]] = None

    # Cardiac/QT
    qtc_ms: Optional[int] = Field(default=None, ge=200, le=800)
    cardiac_history: bool = False

    # Perinatal
    pregnant: bool = False
    postpartum_days: Optional[int] = Field(default=None, ge=0, le=3650)

    # Metabolic
    bmi: Optional[float] = Field(default=None, ge=5, le=120)

    # Comorbidities
    dementia: bool = False
    insomnia: bool = False
    fibromyalgia: bool = False
    suicidality: Literal["none", "elevated_ideation", "acutely_suicidal"] = "none"
    tolerability: Literal["good", "poor"] = "good"   # Step 6a: current trial tolerability
    chronic_suicidality: bool = False                  # elevates lithium to Tier 1 [47, 48, 54]
    prior_depressive_episodes: Optional[int] = None    # None=unknown, 0=first, 1+=recurrent
    discontinuing: bool = False                        # True = tapering to stop (not switching); activates taper schedule in maintenance_plan
    reason_for_switch: Optional[Literal["no_response", "intolerant"]] = None  # Protocol context for switch output
    # Symptom and preference profile (informs report formatter)
    anxiety_comorbidity: bool = False
    sexual_dysfunction_concern: bool = False
    fatigue_anhedonia: bool = False
    therapy_preference: Optional[Literal["medication", "therapy", "combination"]] = None

    @field_validator("current_antidepressant_key")
    @classmethod
    def validate_current_antidepressant_key(cls, v: Optional[str]) -> Optional[str]:
        if v is None:
            return v
        return v.strip().lower()

class EvidenceItem(BaseModel):
    variable: str
    value: str
    fhir_source: str
    note: Optional[str] = None

class MedicationRecommendation(BaseModel):
    medication_key: str
    medication_display: str
    medication_class: str
    intent: RecommendationIntent
    conditional: bool = False
    start_dose: Optional[str] = None
    dose_range: str
    titration_guidance: str
    instructions: List[str] = Field(default_factory=list)
    messages: List[str] = Field(default_factory=list)
    rationale: List[str] = Field(default_factory=list)
    evidence: List[EvidenceItem] = Field(default_factory=list)

class AlgorithmOutput(BaseModel):
    findings: List[str] = Field(default_factory=list)
    recommendations: List[MedicationRecommendation] = Field(default_factory=list)
    non_med_recommendations: List[str] = Field(default_factory=list)
    rationale: List[str] = Field(default_factory=list)
    warnings: List[str] = Field(default_factory=list)
    path_conflicts: List[str] = Field(default_factory=list)   # NEW: inter-path conflicts
    audit_trail: List[str] = Field(default_factory=list)
    stop_reason: Optional[str] = None
    pathway_applied: List[str] = Field(default_factory=lambda: ["adult"])  # CHANGED: list
    monitoring_schedule: List[str] = Field(default_factory=list)
    maintenance_plan: List[str] = Field(default_factory=list)
    medications_excluded: List[str] = Field(default_factory=list)
    adjunctive_options: List[str] = Field(default_factory=list)
    reference_list: List[str] = Field(default_factory=list)
    switching_protocol: List[str] = Field(default_factory=list)
    report: Optional[Any] = Field(default=None)          # structured JSON dict for React
    text_report: Optional[str] = Field(default=None)     # formatted text for notebook display


# ------------------------------------------------------------
# 4) Path-specific rules (Step 2 — deviation paths)
# ------------------------------------------------------------
# Each path defines:
#   exclusions          — {med_key: reason} meds blocked unconditionally by this path
#   qtc_500_exclusions  — {med_key: reason} meds blocked only when QTc > 500 ms
#   dose_caps           — {med_key: max_mg} most restrictive cap across all active paths wins
#   preferred           — [med_key, ...] ordered preferred meds for this path
#
# Applied in ClinicalContraindicationStage; conflicts surfaced via detect_path_conflicts().
PATH_RULES: Dict[str, Dict[str, Any]] = {
    "geriatric": {
        # [26, 28] anticholinergic burden; [26, 28] CYP2D6 interactions / falls
        "exclusions": {
            "paroxetine":  "Anticholinergic burden, falls risk [26, 28]",
            "fluoxetine":  "Drug interactions via CYP2D6, falls risk [26, 28]",
        },
        "qtc_500_exclusions": {},
        # [27, 30] escitalopram max 10 mg; [25, 27] citalopram max 20 mg
        "dose_caps": {
            "escitalopram": 10.0,
            "citalopram":   20.0,
        },
        "preferred": ["sertraline", "escitalopram", "duloxetine"],  # [28]
    },
    "cardiac": {
        "exclusions": {},   # TCAs/trazodone/nefazodone are not in MED_KB; no blanket exclusions
        # [79, 81, 82] QTc > 500 ms: arrhythmia risk
        "qtc_500_exclusions": {
            "citalopram":   "QTc > 500 ms: arrhythmia risk [79, 81, 82]",
            "escitalopram": "QTc > 500 ms: arrhythmia risk [79, 81, 82]",
        },
        "dose_caps": {},
        "preferred": ["sertraline", "fluoxetine"],   # safest cardiac profile [27, 78]
    },
    "renal": {
        # Duloxetine CrCl < 30 handled separately in ClinicalContraindicationStage.
        # Dose caps for specific meds loaded dynamically from RENAL_MAX_CAPS_MG.
        "exclusions": {},
        "qtc_500_exclusions": {},
        "dose_caps": {},
        "preferred": [],
    },
    "hepatic": {
        # [85, 86] duloxetine hepatotoxicity; [85] nefazodone
        "exclusions": {
            "duloxetine":  "Hepatotoxicity risk; avoid entirely in hepatic impairment — do not dose-reduce [85, 86]",
            "nefazodone":  "Hepatotoxic [85]",
        },
        "qtc_500_exclusions": {},
        # [30] escitalopram max 10 mg; citalopram max 20 mg in hepatic impairment
        # [23] venlafaxine 50% dose reduction → max 112.5 mg (50% of 225 mg)
        "dose_caps": {
            "escitalopram":  10.0,
            "citalopram":    20.0,
            "venlafaxine_xr": 112.5,
        },
        "preferred": [],
    },
    "obesity": {
        # [25, 30] significant weight gain
        "exclusions": {
            "mirtazapine": "Significant weight gain; avoid in obesity [25, 30]",
            "paroxetine":  "Significant weight gain; avoid in obesity [25, 30]",
        },
        "qtc_500_exclusions": {},
        "dose_caps": {},
        "preferred": ["bupropion_xl", "fluoxetine"],   # [25]
    },
    "dementia": {
        # [26, 28] anticholinergic burden accelerates cognitive decline
        "exclusions": {
            "paroxetine": "Anticholinergic burden accelerates cognitive decline [26, 28]",
        },
        "qtc_500_exclusions": {},
        "dose_caps": {},
        "preferred": [],
    },
    "antepartum": {
        # [26, 28] cardiac malformations in first-trimester exposure
        "exclusions": {
            "paroxetine": "Associated with cardiac malformations in first-trimester exposure [26, 28]",
        },
        "qtc_500_exclusions": {},
        "dose_caps": {},
        "preferred": ["sertraline", "fluoxetine", "citalopram", "escitalopram"],   # [26, 28]
    },
    "postpartum": {
        # [26, 28] norfluoxetine accumulates in nursing infant
        "exclusions": {
            "fluoxetine": "Long half-life; norfluoxetine accumulates in nursing infant [26, 28]",
        },
        "qtc_500_exclusions": {},
        "dose_caps": {},
        "preferred": ["sertraline"],   # lowest breast milk transfer [26, 28]
    },
    "pain": {
        # [29, 35] SNRIs FDA-approved for pain; SSRIs generally ineffective
        "exclusions": {},
        "qtc_500_exclusions": {},
        "dose_caps": {},
        "preferred": ["duloxetine", "venlafaxine_xr"],   # [29, 35]
    },
    "insomnia": {
        # [25, 30] mirtazapine first-line for insomnia — but only when obesity path is NOT active
        "exclusions": {},
        "qtc_500_exclusions": {},
        "dose_caps": {},
        "preferred": ["mirtazapine"],
    },
    "peds": {
        # [75, 76, 77] venlafaxine highest suicidality signal; [77] paroxetine unfavorable
        "exclusions": {
            "venlafaxine_xr": "Highest suicidality signal in pediatric meta-analyses [75, 76, 77]",
            "paroxetine":     "Not recommended in pediatric population; unfavorable safety profile [77]",
        },
        "qtc_500_exclusions": {},
        "dose_caps": {},
        # [75, 76] fluoxetine FDA-approved age 8+; escitalopram FDA-approved age 12+
        "preferred": ["fluoxetine", "escitalopram", "sertraline"],
    },
    "adult": {
        "exclusions": {},
        "qtc_500_exclusions": {},
        "dose_caps": {},
        # Standard adult SSRI ranking per spec [19, 20, 26]: escitalopram first
        "preferred": ["escitalopram", "sertraline", "fluoxetine", "citalopram", "paroxetine"],
    },
}


# ------------------------------------------------------------
# 5) Multi-path detection — replaces single infer_pathway()
# ------------------------------------------------------------
def detect_active_paths(p: PatientInput) -> List[str]:
    """
    Returns every deviation path triggered for this patient (Step 2 of
    CLAUDE_MDD.md). Multiple paths fire simultaneously when multiple
    triggers are present. Falls back to ['adult'] when no deviation applies.

    Trigger order mirrors Step 2 of CLAUDE_MDD.md.
    """
    paths: List[str] = []

    if p.age < 18:
        paths.append("peds")
    if p.age >= 65:
        paths.append("geriatric")
    if p.pregnant:
        paths.append("antepartum")
    if p.postpartum_days is not None and p.postpartum_days >= 0:
        paths.append("postpartum")

    # Renal trigger: eGFR < 60 per CLAUDE_MDD.md Step 2
    renal_val = p.crcl_ml_min if p.crcl_ml_min is not None else p.egfr_ml_min_1_73
    if renal_val is not None and renal_val < 60:
        paths.append("renal")

    if p.hepatic_impairment:
        paths.append("hepatic")
    if p.cardiac_history or p.qtc_ms is not None:
        paths.append("cardiac")
    if p.fibromyalgia:
        paths.append("pain")
    if p.insomnia:
        paths.append("insomnia")
    if p.bmi is not None and p.bmi >= 30:
        paths.append("obesity")
    if p.dementia:
        paths.append("dementia")

    return paths if paths else ["adult"]


# ------------------------------------------------------------
# 6) Cross-path conflict detection
# ------------------------------------------------------------
def detect_path_conflicts(active_paths: List[str], p: PatientInput) -> List[str]:
    """
    Identifies cases where one path prefers a medication that another path
    excludes. Conflicts are flagged in output.path_conflicts rather than
    silently dropped.
    """
    conflicts: List[str] = []
    for path_a in active_paths:
        prefs_a = PATH_RULES.get(path_a, {}).get("preferred", [])
        for med in prefs_a:
            for path_b in active_paths:
                if path_b == path_a:
                    continue
                rules_b = PATH_RULES.get(path_b, {})
                all_excl_b = dict(rules_b.get("exclusions", {}))
                if p.qtc_ms is not None and p.qtc_ms > 500:
                    all_excl_b.update(rules_b.get("qtc_500_exclusions", {}))
                if med in all_excl_b:
                    msg = (
                        f"Path conflict: [{path_a}] prefers {med} "
                        f"but [{path_b}] excludes it — {all_excl_b[med]}"
                    )
                    if msg not in conflicts:
                        conflicts.append(msg)
    return conflicts


# ------------------------------------------------------------
# 7) Helper functions
# ------------------------------------------------------------
ResponseCategory = Literal["remission_or_response", "partial", "no_response", "insufficient_data"]

def phq_reduction_pct(baseline: int, current: int) -> float:
    if baseline <= 0:
        return 0.0
    return (baseline - current) / baseline

def categorize_response(
    baseline_phq: Optional[int], current_phq: int, weeks_on_med: Optional[int]
) -> ResponseCategory:
    if baseline_phq is None or weeks_on_med is None:
        return "insufficient_data"
    if weeks_on_med < 6:
        return "insufficient_data"
    if current_phq < 5:
        return "remission_or_response"
    pct = phq_reduction_pct(baseline_phq, current_phq)
    if pct >= 0.50:
        return "remission_or_response"
    if 0.25 <= pct < 0.50:
        return "partial"
    return "no_response"

def renal_bucket_from_crcl(crcl: Optional[float], egfr: Optional[float]) -> Optional[str]:
    val = crcl if crcl is not None else egfr
    if val is None:
        return None
    if val >= 60:   return ">60"
    if val >= 30:   return "30-60"
    if val >= 15:   return "15-29"
    return "<15"

def max_cap_for_context(
    p: PatientInput,
    med_key: str,
    active_paths: Optional[List[str]] = None,
) -> Optional[float]:
    """
    Returns the most restrictive dose cap applicable across ALL active paths.
    Returns 0.0 to signal 'avoid entirely'.
    Returns None if no cap applies (use KB default max).
    """
    paths = active_paths if active_paths is not None else detect_active_paths(p)
    caps: List[float] = []

    # Renal caps from RENAL_MAX_CAPS_MG
    if "renal" in paths:
        bucket = renal_bucket_from_crcl(p.crcl_ml_min, p.egfr_ml_min_1_73)
        if bucket and med_key in RENAL_MAX_CAPS_MG:
            caps.append(float(RENAL_MAX_CAPS_MG[med_key][bucket]))

    # Duloxetine: avoid entirely when CrCl/eGFR < 30
    if med_key == "duloxetine":
        renal_val = p.crcl_ml_min if p.crcl_ml_min is not None else p.egfr_ml_min_1_73
        if renal_val is not None and renal_val < 30:
            return 0.0

    # Citalopram: max 20 mg if age > 60 [25, 27] — fires independently of the
    # geriatric path (which triggers at age ≥ 65, not > 60)
    if med_key == "citalopram" and p.age > 60:
        caps.append(20.0)

    # PATH_RULES dose_caps (geriatric, hepatic, etc.) — most restrictive wins
    for path in paths:
        path_caps = PATH_RULES.get(path, {}).get("dose_caps", {})
        if med_key in path_caps:
            caps.append(float(path_caps[med_key]))

    return min(caps) if caps else None

# Hepatic impairment: SSRI start at 50% of standard starting dose [85, 86]
# Venlafaxine start at 50% of standard starting dose [23]
HEPATIC_START_DOSES: Dict[str, str] = {
    "escitalopram":   "2.5–5 mg daily (50% of standard; hepatic impairment [85, 86])",
    "sertraline":     "12.5–25 mg daily (50% of standard; hepatic impairment [85, 86])",
    "citalopram":     "5–10 mg daily (50% of standard; hepatic impairment [85, 86])",
    "fluoxetine":     "5–10 mg daily (50% of standard; hepatic impairment [85, 86])",
    "paroxetine":     "5–10 mg daily (50% of standard; hepatic impairment [85, 86])",
    "fluvoxamine":    "25 mg daily (50% of standard; hepatic impairment [85, 86])",
    "venlafaxine_xr": "~19–37.5 mg daily (50% of standard; hepatic impairment [23])",
}

def get_context_start_dose(
    p: PatientInput,
    med_key: str,
    active_paths: List[str],
) -> Optional[str]:
    """
    Returns a context-adjusted starting dose string when hepatic impairment
    requires dose reduction, or None to use the KB default.
    Applies to all SSRIs and venlafaxine [23, 85, 86].
    """
    if "hepatic" in active_paths and med_key in HEPATIC_START_DOSES:
        return HEPATIC_START_DOSES[med_key]
    return None


def is_qt_risk_med(med_key: str) -> bool:
    return med_key in MED_KB and MED_KB[med_key].get("qt_risk", "none") != "none"

def estimate_trial_number(p: PatientInput) -> int:
    if p.trial_number is not None:
        return int(p.trial_number)
    failed = [t for t in p.prior_trials if t.outcome in {"no_response", "intolerant"}]
    return min(3, 1 + len({t.medication_key for t in failed if t.medication_key}))

def is_at_max_dose_for_context(
    p: PatientInput, med_key: str, active_paths: Optional[List[str]] = None
) -> bool:
    if p.current_dose_mg is None:
        return False
    cap = max_cap_for_context(p, med_key, active_paths)
    if cap == 0.0:
        return False
    typical_max = DOSE_MIN_MAX_MG.get(med_key, (0.0, 1e9))[1]
    max_allowed = typical_max if cap is None else min(typical_max, cap)
    return float(p.current_dose_mg) >= (max_allowed - 1e-6)

def apply_history_sets(p: PatientInput) -> Tuple[Set[str], Set[str], Set[str]]:
    tried      = {t.medication_key.strip().lower() for t in p.prior_trials if t.medication_key}
    failed     = {t.medication_key.strip().lower() for t in p.prior_trials
                  if t.medication_key and t.outcome in {"no_response", "intolerant"}}
    successful = {t.medication_key.strip().lower() for t in p.prior_trials
                  if t.medication_key and t.outcome == "remission_or_response"}
    return tried, failed, successful


# ------------------------------------------------------------
# 8) Pipeline stages
# ------------------------------------------------------------
@dataclass
class WorkingState:
    patient: PatientInput
    output: AlgorithmOutput
    blocked_candidates: Set[str]
    exclusion_reasons: Dict[str, str] = field(default_factory=dict)

class Stage:
    name: str = "stage"
    def run(self, state: WorkingState) -> WorkingState:
        return state

class FindingsStage(Stage):
    name = "findings"

    def run(self, state: WorkingState) -> WorkingState:
        p = state.patient
        out = state.output
        out.findings.append(f"PHQ-9 current: {p.phq_current}.")
        if p.baseline_phq is not None:
            out.findings.append(f"PHQ-9 baseline: {p.baseline_phq}.")
        if p.cardiac_history:
            out.findings.append("Relevant history: cardiac disease.")
        if p.fibromyalgia:
            out.findings.append("Relevant diagnosis: fibromyalgia/chronic pain.")
        if p.insomnia:
            out.findings.append("Relevant symptom: insomnia.")
        if p.hepatic_impairment:
            out.findings.append("Relevant diagnosis: hepatic impairment.")
        if p.dementia:
            out.findings.append("Relevant diagnosis: dementia.")
        if p.crcl_ml_min is not None or p.egfr_ml_min_1_73 is not None:
            rb = renal_bucket_from_crcl(p.crcl_ml_min, p.egfr_ml_min_1_73)
            val = p.crcl_ml_min if p.crcl_ml_min is not None else p.egfr_ml_min_1_73
            out.findings.append(f"Renal function: {val} mL/min (bucket: {rb}).")
        if p.qtc_ms is not None:
            out.findings.append(f"QTc: {p.qtc_ms} ms.")
        if p.pregnant:
            out.findings.append("Population: antepartum.")
        if p.postpartum_days is not None:
            out.findings.append(f"Population: postpartum (day {p.postpartum_days}).")
        if p.bmi is not None:
            out.findings.append(f"BMI: {p.bmi:.1f}.")
        if p.suicidality != "none":
            out.findings.append(f"Suicidality screen: {p.suicidality}.")
        return state

class PathwayStage(Stage):
    """
    Replaces single infer_pathway() with detect_active_paths().
    All triggered deviation paths fire simultaneously.
    """
    name = "pathway"

    def run(self, state: WorkingState) -> WorkingState:
        p = state.patient
        out = state.output
        out.pathway_applied = detect_active_paths(p)
        out.findings.append(f"Active deviation path(s): {', '.join(out.pathway_applied)}.")
        out.audit_trail.append(f"active_paths={out.pathway_applied}")
        return state

class SuicidalityScreenStage(Stage):
    """
    Step 0: Suicidality screen per CLAUDE_MDD.md.

    acutely_suicidal  → hard stop; emergent hospitalization; no further processing.
    elevated_ideation → consider hospitalization flagged in warnings; close
                        monitoring added; algorithm continues.
    age < 25 (any suicidality value) → FDA black box warning always appended.
    Citations: [3, 4, 73, 74]
    """
    name = "suicidality_screen"

    def run(self, state: WorkingState) -> WorkingState:
        p = state.patient
        out = state.output

        # FDA black box warning — fires for all patients under 25 regardless of
        # suicidality value and regardless of any prior stop_reason.
        if p.age < 25:
            out.warnings.append(
                "FDA black box warning: antidepressants increase risk of suicidal "
                "thinking and behavior in patients under 25. Monitor closely during "
                "weeks 1–4. [73, 74]"
            )
            out.audit_trail.append("warn:fda_black_box_age<25")

        # Skip suicidality routing if a prior hard stop is already set
        # (e.g., mania_positive), but black box warning above still fires.
        if out.stop_reason:
            return state

        if p.suicidality == "acutely_suicidal":
            out.stop_reason = "acutely_suicidal"
            out.non_med_recommendations.append(
                "EMERGENT: Patient is acutely suicidal. "
                "Emergent hospitalization is recommended. "
                "No antidepressant initiation at this time. [3, 4, 73, 74]"
            )
            out.warnings.append(
                "Acutely suicidal: emergent evaluation required. "
                "Do not initiate or adjust antidepressants until patient is stabilised "
                "in an appropriate care setting. [3, 4]"
            )
            out.audit_trail.append("stop:acutely_suicidal")

        elif p.suicidality == "elevated_ideation":
            out.warnings.append(
                "Elevated suicidal ideation (without acute risk): consider hospitalisation; "
                "initiate close monitoring. Algorithm continues — review all recommendations "
                "in context of suicide risk. [3, 4, 73, 74]"
            )
            out.non_med_recommendations.append(
                "Close suicidality monitoring required: reassess at every visit; "
                "safety plan recommended; consider Crisis/988 referral. [73, 74]"
            )
            out.audit_trail.append("warn:elevated_suicidal_ideation")

        return state


class ManiaExclusionStage(Stage):
    name = "mania_exclusion"

    def run(self, state: WorkingState) -> WorkingState:
        p = state.patient
        out = state.output
        if p.mania_screen == "positive":
            out.stop_reason = "mania_positive"
            out.rationale.append("Mania screen positive: antidepressants not recommended; route to psychiatry. [3, 4]")
            out.non_med_recommendations.append("Do NOT initiate antidepressant; refer for psychiatric evaluation.")
            out.audit_trail.append("stop:mania_positive")
        return state

class QTcSafetyStage(Stage):
    name = "qtc_safety"

    def run(self, state: WorkingState) -> WorkingState:
        p = state.patient
        out = state.output
        if p.qtc_ms is not None and p.qtc_ms > 550:
            out.stop_reason = "qtc_over_550"
            out.warnings.append("QTc > 550 ms: discontinue QT-prolonging agents and obtain psychiatric consultation per protocol.")
            out.audit_trail.append("stop:qtc_over_550")
        elif p.qtc_ms is not None and p.qtc_ms >= 500:
            out.warnings.append("QTc ≥ 500 ms: elevated risk — QTc-conditional exclusions active; clinician review advised.")
            out.audit_trail.append("warn:qtc_ge_500")
        return state

class SeverityAndNonMedStage(Stage):
    name = "severity_and_non_med"

    def run(self, state: WorkingState) -> WorkingState:
        p = state.patient
        out = state.output
        followup_context = (
            p.baseline_phq is not None
            and p.weeks_on_current_antidepressant is not None
            and p.weeks_on_current_antidepressant >= 6
            and p.current_antidepressant_key is not None
            and p.current_antidepressant_key.strip() != ""
        )
        out.audit_trail.append(f"followup_context={followup_context}")

        if p.phq_current < 10 and not followup_context:
            if p.phq_current < 5:
                out.non_med_recommendations.append("Monitoring only (PHQ < 5).")
            else:
                out.non_med_recommendations.append("Recommend MoodCalmer dCBT (PHQ 5–9).")
            out.rationale.append("No medication recommended when PHQ < 10 in a new-start context.")
            out.audit_trail.append("no_meds_phq<10_new_start")
            return state

        if p.phq_current >= 15:
            out.non_med_recommendations.append("Recommend psychotherapy in combination with pharmacotherapy (PHQ ≥ 15). [3, 4]")
            out.audit_trail.append("psychotherapy_phq>=15")
        if 10 <= p.phq_current <= 14 and not followup_context:
            out.non_med_recommendations.append(
                "Shared decision (PHQ 10–14): psychotherapy alone first-line; consider medication if "
                "psychotherapy fails, patient preference, or history of recurrent depression. [1, 2, 3, 4]"
            )
            out.audit_trail.append("shared_decision_phq10-14")
        return state

class MedSafetyStage(Stage):
    name = "med_safety"

    def run(self, state: WorkingState) -> WorkingState:
        p = state.patient
        out = state.output
        if out.stop_reason:
            return state
        safety = medication_safety_screen(p.current_medications, list(MED_KB.keys()))
        out.warnings.extend(safety.warnings)
        out.audit_trail.append(f"med_safety_blocked={sorted(safety.blocked)}")
        if safety.detected:
            out.audit_trail.append(f"med_safety_signals={list(safety.detected.keys())}")
        state.blocked_candidates |= safety.blocked
        return state

class ClinicalContraindicationStage(Stage):
    """
    Iterates over every active deviation path and applies:
      - Unconditional exclusions (PATH_RULES[path]['exclusions'])
      - QTc-conditional exclusions (PATH_RULES[path]['qtc_500_exclusions']) when QTc > 500
      - Dose-cap warnings for the current medication context
    Each blocked medication is stored in state.exclusion_reasons for
    structured output. Cross-path conflicts are detected and surfaced.
    """
    name = "clinical_contraindications"

    def run(self, state: WorkingState) -> WorkingState:
        p = state.patient
        out = state.output
        active_paths = out.pathway_applied

        for path in active_paths:
            rules = PATH_RULES.get(path, {})

            # Unconditional exclusions
            for med, reason in rules.get("exclusions", {}).items():
                if med not in state.blocked_candidates:
                    state.blocked_candidates.add(med)
                    state.exclusion_reasons[med] = f"[{path}] {reason}"
                    out.audit_trail.append(f"blocked:{med} ({path})")

            # QTc-conditional exclusions (only when QTc > 500 ms)
            if p.qtc_ms is not None and p.qtc_ms > 500:
                for med, reason in rules.get("qtc_500_exclusions", {}).items():
                    if med not in state.blocked_candidates:
                        state.blocked_candidates.add(med)
                        state.exclusion_reasons[med] = f"[{path}, QTc > 500 ms] {reason}"
                        out.audit_trail.append(f"blocked:{med} (qtc>500, {path})")

            # Dose-cap warnings (note: the cap is enforced in max_cap_for_context;
            # this just surfaces it for clinician awareness)
            for med, cap in rules.get("dose_caps", {}).items():
                if med in MED_KB:
                    out.warnings.append(
                        f"[{path}] {MED_KB[med]['display']}: "
                        f"dose cap {cap} mg/day in this context."
                    )

        # Special renal rule: duloxetine avoid if CrCl/eGFR < 30
        renal_val = p.crcl_ml_min if p.crcl_ml_min is not None else p.egfr_ml_min_1_73
        if renal_val is not None and renal_val < 30 and "duloxetine" not in state.blocked_candidates:
            state.blocked_candidates.add("duloxetine")
            state.exclusion_reasons["duloxetine"] = "[renal] CrCl/eGFR < 30: avoid duloxetine [85, 86]"
            out.audit_trail.append("blocked:duloxetine_renal<30")

        # Dementia + SGA black box warning
        if p.dementia:
            out.warnings.append(
                "Dementia present: all atypical antipsychotics carry FDA black box warning "
                "for increased mortality in elderly patients — flag if augmentation is considered [26, 28]."
            )

        # Cross-path conflict detection
        conflicts = detect_path_conflicts(active_paths, p)
        out.path_conflicts.extend(conflicts)
        if conflicts:
            out.audit_trail.append(f"path_conflicts={len(conflicts)}")

        # Append structured exclusions to findings
        for med, reason in state.exclusion_reasons.items():
            display = MED_KB[med]["display"] if med in MED_KB else med
            out.findings.append(f"Excluded: {display} — {reason}")

        return state

class DoseSanityStage(Stage):
    name = "dose_sanity"

    def run(self, state: WorkingState) -> WorkingState:
        p = state.patient
        out = state.output
        active_paths = out.pathway_applied

        if not p.current_antidepressant_key or p.current_dose_mg is None:
            out.audit_trail.append("dose_sanity_skipped")
            return state

        med = p.current_antidepressant_key.strip().lower()
        if med not in DOSE_MIN_MAX_MG:
            out.audit_trail.append(f"dose_sanity_no_bounds:{med}")
            return state

        cap = max_cap_for_context(p, med, active_paths)
        if cap == 0.0:
            out.warnings.append(f"{med} is contraindicated in current context. Clinician review required.")
            out.audit_trail.append("dose_sanity_contraindicated")
            return state

        mn, mx = DOSE_MIN_MAX_MG[med]
        if cap is not None:
            mx = min(mx, float(cap))
        dose = float(p.current_dose_mg)
        if dose < mn or dose > mx:
            out.warnings.append(
                f"Current {med} dose ({dose} mg) is outside context range ({mn}–{mx} mg). "
                f"Verify before titration decisions."
            )
            out.audit_trail.append("dose_sanity_out_of_range")
        else:
            out.audit_trail.append("dose_sanity_ok")
        return state


# ------------------------------------------------------------
# 9) Recommendation builder
# ------------------------------------------------------------
def _make_reco(
    p: PatientInput,
    med_key: str,
    intent: RecommendationIntent,
    active_paths: Optional[List[str]] = None,
    additional_messages: Optional[List[str]] = None,
    rationale: Optional[List[str]] = None,
    evidence: Optional[List[EvidenceItem]] = None,
    conditional: bool = False,
    cap_override: Optional[float] = None,
) -> MedicationRecommendation:
    kb = MED_KB[med_key]
    msgs = list(additional_messages or [])
    rat  = list(rationale or [])
    ev   = list(evidence or [])
    paths = active_paths if active_paths is not None else detect_active_paths(p)

    if kb.get("bp_message_relevant", False):
        msgs.append("Check baseline BP before prescribing; monitor during titration.")

    if p.qtc_ms is None and is_qt_risk_med(med_key):
        msgs.append("QT risk: prescribe only after obtaining baseline QTc/ECG.")
        conditional = True
        ev.append(EvidenceItem(variable="QTc", value="missing", fhir_source="Observation"))

    if p.qtc_ms is not None and p.qtc_ms >= 500 and is_qt_risk_med(med_key):
        msgs.append("QTc elevated: avoid stacking QT-prolonging agents; clinician review.")
        ev.append(EvidenceItem(variable="QTc", value=str(p.qtc_ms), fhir_source="Observation"))

    for m in kb.get("monitoring", []):
        if m not in msgs:
            msgs.append(m)

    # Determine effective cap across all active paths
    effective_cap = cap_override if cap_override is not None else max_cap_for_context(p, med_key, paths)
    if effective_cap is not None and med_key in DOSE_MIN_MAX_MG and effective_cap > 0:
        mn, mx_typ = DOSE_MIN_MAX_MG[med_key]
        mx_ctx = min(mx_typ, float(effective_cap))
        msgs.append(f"Max dose for this patient context (all active paths): {mx_ctx} mg/day.")

    # Use context-adjusted starting dose when hepatic path requires reduction
    if intent in {"start", "switch_to", "augment_with"}:
        ctx_start = get_context_start_dose(p, med_key, paths)
        start_dose = ctx_start if ctx_start is not None else kb["start_dose"]
    else:
        start_dose = None

    return MedicationRecommendation(
        medication_key=med_key,
        medication_display=kb["display"],
        medication_class=kb["class"],
        intent=intent,
        conditional=conditional,
        start_dose=start_dose,
        dose_range=kb["dose_range"],
        titration_guidance=kb["titration"],
        instructions=[],
        messages=msgs,
        rationale=rat,
        evidence=ev,
    )


# ------------------------------------------------------------
# 10) Treatment selection stage
# ------------------------------------------------------------


# ============================================================
# Reference Database — loaded at runtime from references_consolidated.md
# ============================================================
REFERENCE_DB: Dict[str, str] = {}
_ref_path = Path("/Users/matthewmiclette/Documents/Projects/references_consolidated.md")
with open(_ref_path) as _ref_f:
    for _ref_line in _ref_f:
        _ref_m = re.match(r'^(\d+)\. (.+)', _ref_line.strip())
        if _ref_m:
            REFERENCE_DB[_ref_m.group(1)] = f"{_ref_m.group(1)}. {_ref_m.group(2)}"



# ============================================================
# Augmentation helpers — Step 6 / 6a / 6b
# ============================================================

SSRI_KEYS  = frozenset({"sertraline", "escitalopram", "citalopram", "fluoxetine",
                         "paroxetine", "fluvoxamine"})
SNRI_KEYS  = frozenset({"venlafaxine_xr", "duloxetine", "desvenlafaxine",
                         "levomilnacipran_er"})
BUP_KEYS   = frozenset({"bupropion_xl"})
MIRT_KEYS  = frozenset({"mirtazapine"})

# High discontinuation syndrome risk — extended taper required [28, 87, 92, 93, 94]
EXTENDED_TAPER_MEDS: frozenset = frozenset({
    "paroxetine", "venlafaxine_xr", "duloxetine", "desvenlafaxine"
})

# Serotonergic antidepressants — used for multi-agent serotonin syndrome check [89, 90, 98]
SEROTONERGIC_AD_KEYS: frozenset = frozenset({
    "sertraline", "escitalopram", "citalopram", "fluoxetine", "paroxetine",
    "fluvoxamine", "venlafaxine_xr", "duloxetine", "desvenlafaxine",
    "levomilnacipran_er", "vortioxetine", "vilazodone",
    "mirtazapine",
})

# MAOI antidepressants — 2-week washout required before/after [28, 87, 89, 90]
MAOI_AD_KEYS: frozenset = frozenset({
    "phenelzine", "tranylcypromine", "isocarboxazid", "selegiline",
})

# Tricyclic antidepressants — cross-taper 1-2 weeks when switching from SSRI/SNRI [28, 87]
TCA_AD_KEYS: frozenset = frozenset({
    "nortriptyline", "desipramine", "amitriptyline", "imipramine",
    "clomipramine", "doxepin",
})

# Atypical antipsychotics in order of preference per [44]
SGA_AUGMENT_ORDER = [
    "aripiprazole",   # first choice — strongest evidence; superior efficacy/acceptability [44]
    "brexpiprazole",  # [44]
    "quetiapine",     # 150–300 mg/day [44, 54]
    "cariprazine",    # [44]
    "risperidone",    # superior efficacy/acceptability with aripiprazole [44]
    "olanzapine",     # flag combination pill [44]
]


def current_med_class(med_key: Optional[str]) -> str:
    """Classify current antidepressant for augmentation tier-1 logic."""
    if not med_key:
        return "unknown"
    k = med_key.strip().lower()
    if k in SSRI_KEYS:  return "ssri"
    if k in SNRI_KEYS:  return "snri"
    if k in BUP_KEYS:   return "bupropion"
    if k in MIRT_KEYS:  return "mirtazapine"
    return "other"


# ── Switching protocol lookup ─────────────────────────────────────────────
@dataclass
class SwitchProtocol:
    method: str         # "direct_switch" | "cross_taper" | "slow_taper" | "washout"
    duration: str       # "N/A" | "1–2 weeks" | "2–4 weeks minimum" | etc.
    warning: str        # "" or warning text (no trailing citation)
    citations: List[str]

METHOD_DISPLAY: Dict[str, str] = {
    "direct_switch": "Direct switch",
    "cross_taper":   "Cross-taper",
    "slow_taper":    "Slow taper",
    "washout":       "Washout — do not cross-taper",
}

def classify_med_for_switch(med_key: Optional[str]) -> str:
    """Return a switch-class token for a medication key."""
    if not med_key:
        return "unknown"
    k = med_key.lower().strip()
    if k == "fluoxetine":    return "fluoxetine"
    if k == "paroxetine":    return "paroxetine"
    if k == "fluvoxamine":   return "fluvoxamine"
    if k in SSRI_KEYS:       return "ssri_other"    # sertraline/escitalopram/citalopram
    if k == "venlafaxine_xr": return "venlafaxine"
    if k == "duloxetine":    return "duloxetine"
    if k in SNRI_KEYS:       return "snri_other"    # desvenlafaxine / levomilnacipran
    if k in BUP_KEYS:        return "bupropion"
    if k in MIRT_KEYS:       return "mirtazapine"
    if k in TCA_AD_KEYS:     return "tca"
    if k in MAOI_AD_KEYS:    return "maoi"
    return "other"

def _cls_is_ssri(c: str) -> bool:
    return c in {"fluoxetine", "paroxetine", "fluvoxamine", "ssri_other"}

def _cls_is_snri(c: str) -> bool:
    return c in {"venlafaxine", "duloxetine", "snri_other"}

def _cls_is_ssri_or_snri(c: str) -> bool:
    return _cls_is_ssri(c) or _cls_is_snri(c)

def _cls_is_bup_or_mirt(c: str) -> bool:
    return c in {"bupropion", "mirtazapine"}


def get_switching_protocol(prior_key: Optional[str], new_key: Optional[str]) -> SwitchProtocol:
    """
    Returns the appropriate class-to-class switching protocol.
    Rules are applied in priority order (most specific first).
    Covers every combination required by Step 7a of CLAUDE_MDD.md.
    """
    pc = classify_med_for_switch(prior_key)
    nc = classify_med_for_switch(new_key)

    # ── Priority 1: fluoxetine → MAOI (5-week washout — more restrictive) ─
    if pc == "fluoxetine" and nc == "maoi":
        return SwitchProtocol(
            method="washout",
            duration="5-week washout required before starting MAOI",
            warning="Life-threatening serotonin syndrome risk if washout not observed.",
            citations=["28", "87", "89", "90"],
        )

    # ── Priority 2: MAOI → any (2-week washout after stopping MAOI) ───────
    if pc == "maoi":
        return SwitchProtocol(
            method="washout",
            duration="2-week washout minimum after stopping MAOI",
            warning="Life-threatening serotonin syndrome risk if new antidepressant started too soon.",
            citations=["28", "87", "89", "90"],
        )

    # ── Priority 3: any (except fluoxetine) → MAOI (2-week washout) ───────
    if nc == "maoi":
        return SwitchProtocol(
            method="washout",
            duration="2-week washout minimum before starting MAOI",
            warning="Life-threatening serotonin syndrome risk if washout not observed.",
            citations=["28", "87", "89", "90"],
        )

    # ── Priority 4: paroxetine → any (cross-taper, extended taper) ─────────
    if pc == "paroxetine":
        return SwitchProtocol(
            method="cross_taper",
            duration="1–2 weeks",
            warning=(
                "Extended taper risk — reduce paroxetine 10 mg every 3–7 days while "
                "starting new agent at low dose simultaneously. Abrupt discontinuation "
                "causes symptoms in 30–50% of patients."
            ),
            citations=["28", "87", "93", "94"],
        )

    # ── Priority 5: fluvoxamine → any (cross-taper) ────────────────────────
    if pc == "fluvoxamine":
        return SwitchProtocol(
            method="cross_taper",
            duration="1–2 weeks",
            warning="Discontinuation risk; cross-taper to minimise symptoms.",
            citations=["28", "87"],
        )

    # ── Priority 6: venlafaxine → any (slow taper, highest risk) ───────────
    if pc == "venlafaxine":
        return SwitchProtocol(
            method="slow_taper",
            duration="2–4 weeks minimum",
            warning=(
                "Highest discontinuation syndrome risk — do not switch abruptly. "
                "Symptoms onset within 2–4 days of stopping. "
                "Cross-taper with overlap recommended."
            ),
            citations=["28", "87", "92", "93", "94"],
        )

    # ── Priority 7: duloxetine → any (slow taper) ──────────────────────────
    if pc == "duloxetine":
        return SwitchProtocol(
            method="slow_taper",
            duration="2–4 weeks minimum",
            warning=(
                "High discontinuation syndrome risk — cross-taper with overlap recommended."
            ),
            citations=["28", "87", "93", "94"],
        )

    # ── Priority 8: fluoxetine → any non-MAOI (direct switch) ──────────────
    if pc == "fluoxetine":
        return SwitchProtocol(
            method="direct_switch",
            duration="N/A",
            warning="Norfluoxetine metabolite (t½ 7–15 days) provides built-in taper.",
            citations=["28", "87"],
        )

    # ── Priority 9: SSRI/SNRI → bupropion or mirtazapine (cross-taper) ─────
    if _cls_is_ssri_or_snri(pc) and _cls_is_bup_or_mirt(nc):
        return SwitchProtocol(
            method="cross_taper",
            duration="1–2 weeks",
            warning="Monitor for serotonin syndrome during overlap period.",
            citations=["28", "87", "89"],
        )

    # ── Priority 10: SSRI/SNRI → TCA (cross-taper) ─────────────────────────
    if _cls_is_ssri_or_snri(pc) and nc == "tca":
        return SwitchProtocol(
            method="cross_taper",
            duration="1–2 weeks",
            warning="Monitor for serotonin syndrome during overlap period.",
            citations=["28", "87", "89"],
        )

    # ── Priority 11: bupropion/mirtazapine → SSRI/SNRI (cross-taper) ───────
    if _cls_is_bup_or_mirt(pc) and _cls_is_ssri_or_snri(nc):
        return SwitchProtocol(
            method="cross_taper",
            duration="1–2 weeks",
            warning="",
            citations=["28", "87"],
        )

    # ── Priority 12: ssri_other → SSRI or SNRI (direct switch) ─────────────
    if pc == "ssri_other" and (_cls_is_ssri(nc) or _cls_is_snri(nc)):
        return SwitchProtocol(
            method="direct_switch",
            duration="N/A",
            warning="",
            citations=["28", "87"],
        )

    # ── Default: cross-taper 1–2 weeks ─────────────────────────────────────
    return SwitchProtocol(
        method="cross_taper",
        duration="1–2 weeks",
        warning="",
        citations=["28", "87"],
    )


def _format_switch_protocol_lines(
    prior_key: Optional[str],
    new_key: Optional[str],
    proto: SwitchProtocol,
) -> List[str]:
    """Format one switching protocol block as a list of strings for output."""
    prior_disp = MED_KB.get(prior_key or "", {}).get("display", prior_key or "unknown")
    new_disp   = MED_KB.get(new_key   or "", {}).get("display", new_key   or "unknown")
    prior_cls  = MED_KB.get(prior_key or "", {}).get("class", "unknown")
    new_cls    = MED_KB.get(new_key   or "", {}).get("class", "unknown")
    ref_str    = ", ".join(f"[{c}]" for c in proto.citations)
    method_str = METHOD_DISPLAY.get(proto.method, proto.method)
    lines = [
        f"Prior medication: {prior_disp} ({prior_cls})",
        f"New medication:   {new_disp} ({new_cls})",
        f"Method: {method_str}",
        f"Duration: {proto.duration}",
    ]
    if proto.warning:
        lines.append(f"Warning: {proto.warning} {ref_str}")
    else:
        lines.append(f"References: {ref_str}")
    return lines


def _switch_taper_message(outgoing_key: Optional[str], incoming_key: Optional[str] = None) -> str:
    """Return a compact one-line taper message for switch_to recommendation messages."""
    proto = get_switching_protocol(outgoing_key, incoming_key)
    method_str = METHOD_DISPLAY.get(proto.method, proto.method)
    ref_str = ", ".join(f"[{c}]" for c in proto.citations)
    if proto.warning:
        return f"{method_str} ({proto.duration}): {proto.warning} {ref_str}"
    return f"{method_str} ({proto.duration}). {ref_str}"



def build_augmentation_recs(
    p: PatientInput,
    current_med: Optional[str],
    pathways: List[str],
    can_rec: Callable[[str], bool],
    state: "WorkingState",
) -> List[MedicationRecommendation]:
    """
    Four-tier augmentation hierarchy per Steps 6, 6a, 6b of CLAUDE_MDD.md.

    Tier 1 — Co-equal first choices: aripiprazole [43, 45, 46] +
               bupropion (if on SSRI/SNRI) [45, 46].
               Exception: chronic suicidality → lithium elevated to Tier 1 [47, 48, 54].
               NOTE: mirtazapine removed from Tier 1 — no clinically meaningful benefit [43, 51].
    Tier 2 — Second-choice SGAs after Tier 1 failure: quetiapine, brexpiprazole,
               cariprazine, lurasidone [24, 43, 44, 49, 50, 51].
    Tier 3 — Targeted: lithium (TRD/after SGAs) [47, 48, 49, 50, 54],
               methylphenidate (residual anergia) [56, 57, 58],
               lisdexamfetamine (off-label) [43, 59].
    """
    out = state.output
    recs: List[MedicationRecommendation] = []
    cls = current_med_class(current_med)
    is_chronic_suicidal = getattr(p, "chronic_suicidality", False)

    # ── Mirtazapine note ─────────────────────────────────────────────────────────────────
    out.rationale.append(
        "NOTE: mirtazapine is NOT a first-choice augmentation option. High-quality evidence "
        "shows no clinically meaningful benefit (MD on BDI-II −1.7, 95% CI −4.03 to 0.63) "
        "and significantly higher discontinuation rates than placebo [43, 51]."
    )

    # ── Dementia warning (applies to all SGAs) ───────────────────────────────────────────────────────
    if p.dementia:
        out.warnings.append(
            "FDA black box warning: all atypical antipsychotics carry increased mortality "
            "risk in elderly patients with dementia — use only after careful benefit-risk "
            "assessment [26, 28]."
        )

    # ── Tier 1: Co-equal first choices ─────────────────────────────────────────────────────────────────

    # Exception: chronic suicidality → lithium first
    if is_chronic_suicidal and can_rec("lithium"):
        recs.append(_make_reco(
            p, "lithium", intent="augment_with", active_paths=pathways,
            rationale=[
                "FIRST CHOICE augmentation (chronic suicidality) — lithium: unique anti-suicide "
                "properties; elevated to Tier 1 when chronic suicidality present [54, 47, 48]."
            ],
            additional_messages=[
                "Target: 0.6–1.2 mEq/L general; 0.3–0.6 mmol/L older adults [54, 54].",
                "Monitor: Li+ level, TSH, BMP at initiation, 1–2 months, then every 6–12 months.",
            ],
            evidence=[EvidenceItem(
                variable="Augmentation tier",
                value="1 — lithium (chronic suicidality exception)",
                fhir_source="Derived",
            )],
        ))

    # Aripiprazole — Tier 1 for all patients
    if can_rec("aripiprazole"):
        recs.append(_make_reco(
            p, "aripiprazole", intent="augment_with", active_paths=pathways,
            rationale=[
                "FIRST CHOICE augmentation — aripiprazole: NNT=8 for response. "
                "OPTIMUM trial: 28.9% remission vs 19.3% for switching [46, 45, 43]."
            ],
            evidence=[EvidenceItem(
                variable="Augmentation tier",
                value="1 — aripiprazole",
                fhir_source="Derived",
            )],
        ))

    # Bupropion — Tier 1 only if on SSRI or SNRI
    if cls in {"ssri", "snri"} and can_rec("bupropion_xl"):
        recs.append(_make_reco(
            p, "bupropion_xl", intent="augment_with", active_paths=pathways,
            rationale=[
                "FIRST CHOICE augmentation — add bupropion (patient on "
                f"{'SSRI' if cls == 'ssri' else 'SNRI'}): comparable remission to aripiprazole "
                "(28.2% vs 28.9% in OPTIMUM trial); complementary dopamine-norepinephrine "
                "reuptake inhibition [45, 46]."
            ],
            evidence=[EvidenceItem(
                variable="Augmentation tier",
                value="1 — bupropion (SSRI/SNRI combination)",
                fhir_source="Derived",
            )],
        ))

    # ── Tier 2: Second-choice SGAs ───────────────────────────────────────────────────────────────────
    tier2 = [
        ("quetiapine",    "SMD −0.32 efficacy; higher discontinuation (RR 1.57); "
                          "2025 LQD trial: superior to lithium at 52 weeks [51, 49, 50]."),
        ("brexpiprazole", "FDA-approved for MDD augmentation; OR 1.47–2.17 for response [24, 43, 44]."),
        ("cariprazine",   "FDA-approved for MDD augmentation; OR 1.34 for response [24, 43, 44]."),
        ("lurasidone",    "Limited evidence in unipolar depression; most data in bipolar — "
                          "use cautiously [43]."),
    ]
    for m, note in tier2:
        if can_rec(m):
            recs.append(_make_reco(
                p, m, intent="augment_with", active_paths=pathways,
                rationale=[f"SECOND CHOICE augmentation (after Tier 1 failure) — {note}"],
                evidence=[EvidenceItem(
                    variable="Augmentation tier",
                    value="2 — atypical antipsychotic",
                    fhir_source="Derived",
                )],
            ))

    # ── Tier 3: Targeted options ─────────────────────────────────────────────────────────────────────────
    if not is_chronic_suicidal and can_rec("lithium"):
        recs.append(_make_reco(
            p, "lithium", intent="augment_with", active_paths=pathways,
            rationale=[
                "THIRD CHOICE augmentation — lithium: reserve for TRD or after SGAs fail. "
                "STAR*D: only 15.9% remission; 2025 LQD trial found quetiapine superior "
                "[49, 50, 54, 47, 48]."
            ],
            additional_messages=[
                "Target: 0.6–1.2 mEq/L general; 0.3–0.6 mmol/L older adults [54, 54].",
                "Monitor: Li+ level, TSH, BMP at initiation, 1–2 months, then every 6–12 months.",
            ],
            evidence=[EvidenceItem(
                variable="Augmentation tier",
                value="3 — lithium (TRD / after SGAs fail)",
                fhir_source="Derived",
            )],
        ))

    if can_rec("methylphenidate"):
        recs.append(_make_reco(
            p, "methylphenidate", intent="augment_with", active_paths=pathways,
            rationale=[
                "THIRD CHOICE augmentation — methylphenidate: residual anergia or fatigue "
                "only; low-to-very-low evidence quality; not for broad augmentation [56, 57, 58]."
            ],
            evidence=[EvidenceItem(
                variable="Augmentation tier",
                value="3 — stimulant (residual anergia only)",
                fhir_source="Derived",
            )],
        ))

    if can_rec("lisdexamfetamine"):
        recs.append(_make_reco(
            p, "lisdexamfetamine", intent="augment_with", active_paths=pathways,
            rationale=[
                "THIRD CHOICE augmentation — lisdexamfetamine: one positive RCT; "
                "FDA-approved for binge eating disorder only — off-label for MDD [43, 59]."
            ],
            evidence=[EvidenceItem(
                variable="Augmentation tier",
                value="3 — lisdexamfetamine (off-label for MDD)",
                fhir_source="Derived",
            )],
        ))

    return recs

class TreatmentSelectionStage(Stage):
    name = "treatment_selection"

    def run(self, state: WorkingState) -> WorkingState:
        p = state.patient
        out = state.output

        if out.stop_reason:
            out.audit_trail.append("selection_skipped:stop_reason")
            return state

        pathways: List[str] = out.pathway_applied   # now a list
        resp_cat = categorize_response(p.baseline_phq, p.phq_current, p.weeks_on_current_antidepressant)
        out.audit_trail.append(f"response_category={resp_cat}")
        trial_n = estimate_trial_number(p)
        out.audit_trail.append(f"trial_number={trial_n}")

        tried_meds, failed_meds, successful_meds = apply_history_sets(p)
        blocked = set(state.blocked_candidates)

        def can_recommend(m: str) -> bool:
            if m not in MED_KB:                          return False
            if m in blocked:                             return False
            if m in failed_meds:                         return False
            if m in tried_meds and m not in successful_meds: return False
            if max_cap_for_context(p, m, pathways) == 0.0:  return False
            return True

        current_med = (p.current_antidepressant_key or "").strip().lower() or None

        # --------------------------------------------------------
        # Pool builders — merge preferred lists from all active paths
        # --------------------------------------------------------
        def first_line_pool() -> List[str]:
            # Perinatal constrains the pool absolutely
            if "antepartum" in pathways or "postpartum" in pathways:
                return ["sertraline", "escitalopram", "fluoxetine", "citalopram"]

            # Pediatric pool (age-gated within peds path)
            if "peds" in pathways:
                if p.age < 12:
                    return ["fluoxetine"]
                return ["fluoxetine", "escitalopram", "sertraline"]

            seen: Set[str] = set()
            merged: List[str] = []

            def add(m: str) -> None:
                if m not in seen:
                    merged.append(m)
                    seen.add(m)

            # Pain path anchors first (SNRI-first mandate)
            if "pain" in pathways:
                for m in PATH_RULES["pain"]["preferred"]:
                    add(m)

            # Insomnia path: mirtazapine only when obesity path is NOT also active
            # (insomnia + obesity conflict → obesity exclusion wins, noted in rationale)
            if "insomnia" in pathways and "obesity" not in pathways:
                for m in PATH_RULES["insomnia"]["preferred"]:
                    add(m)

            # All remaining active paths contribute their preferred lists
            for path in pathways:
                if path in {"pain", "insomnia", "renal", "adult"}:
                    continue   # pain/insomnia handled above; renal/adult have no preferred list
                for m in PATH_RULES.get(path, {}).get("preferred", []):
                    add(m)

            # Standard adult pool as fallback (escitalopram first per spec [19, 20, 26])
            for m in ["escitalopram", "sertraline", "fluoxetine", "citalopram", "paroxetine"]:
                add(m)

            return merged

        def second_line_pool() -> List[str]:
            if "antepartum" in pathways or "postpartum" in pathways:
                return ["sertraline", "escitalopram", "fluoxetine"]
            base = first_line_pool()
            seen_base = set(base)
            snris = [m for m in ["venlafaxine_xr", "duloxetine", "desvenlafaxine", "levomilnacipran_er"]
                     if m not in seen_base]
            return base + snris

        def third_line_pool() -> List[str]:
            return [
                "venlafaxine_xr", "duloxetine", "desvenlafaxine", "levomilnacipran_er",
                "bupropion_xl", "mirtazapine", "vortioxetine", "vilazodone",
            ]

        def pick_two(pool: List[str]) -> List[str]:
            picked: List[str] = []
            for m in pool:
                if can_recommend(m) and m not in picked:
                    picked.append(m)
                if len(picked) == 2:
                    break
            return picked

        # --------------------------------------------------------
        # Follow-up logic
        # --------------------------------------------------------
        if resp_cat in {"partial", "no_response", "remission_or_response"} and current_med in MED_KB:
            if resp_cat == "remission_or_response":
                cap = max_cap_for_context(p, current_med, pathways)
                out.recommendations = [_make_reco(
                    p, current_med, intent="continue", active_paths=pathways,
                    additional_messages=["Response/remission achieved: continue current regimen and monitor."],
                    rationale=["PHQ improvement meets response/remission threshold at ≥6 weeks."],
                    evidence=[
                        EvidenceItem(variable="PHQ-9 current", value=str(p.phq_current), fhir_source="Observation"),
                        EvidenceItem(variable="PHQ-9 baseline", value=str(p.baseline_phq), fhir_source="Derived"),
                        EvidenceItem(variable="Weeks on med", value=str(p.weeks_on_current_antidepressant), fhir_source="Derived"),
                    ],
                    cap_override=cap,
                )]
                out.rationale.append("Continue effective medication when response/remission achieved.")
                return state

            if resp_cat == "partial":
                cap = max_cap_for_context(p, current_med, pathways)
                if not is_at_max_dose_for_context(p, current_med, pathways):
                    inc_msg = "Partial response: increase dose as tolerated; reassess in ~4–6 weeks."
                    if p.current_dose_mg is not None:
                        inc_msg = (
                            f"Partial response: "
                            f"{next_dose_step(current_med, float(p.current_dose_mg), cap_override=cap)} "
                            f"Reassess in ~4–6 weeks."
                        )
                    out.recommendations = [_make_reco(
                        p, current_med, intent="increase", active_paths=pathways,
                        additional_messages=[inc_msg],
                        rationale=["Partial response at ≥6 weeks; room to increase dose in this context."],
                        evidence=[
                            EvidenceItem(variable="Weeks on med", value=str(p.weeks_on_current_antidepressant), fhir_source="Derived"),
                            EvidenceItem(variable="Current dose (mg)", value=str(p.current_dose_mg), fhir_source="MedicationRequest"),
                        ],
                        cap_override=cap,
                    )]
                    out.rationale.append("Titrate before augmenting when partial response and room remains.")
                    return state

                # At max dose — three-tier augmentation per Steps 6/6a/6b of CLAUDE_MDD.md
                continue_reco = _make_reco(
                    p, current_med, intent="continue", active_paths=pathways,
                    additional_messages=["Partial at max dose: continue and add augmenter."],
                    rationale=[
                        "Partial response at maximally tolerated dose → augmentation pathway "
                        "[36, 37, 38, 39, 40, 47, 48, 25, 45, 41]."
                    ],
                    evidence=[
                        EvidenceItem(variable="Trial number", value=str(trial_n), fhir_source="Derived"),
                        EvidenceItem(variable="At max dose", value="true", fhir_source="Derived"),
                    ],
                    cap_override=max_cap_for_context(p, current_med, pathways),
                )

                if trial_n == 3:
                    # Step 6b: third-trial partial response
                    out.non_med_recommendations.append(
                        "Step 6b — Psychiatric consultation advisable: third-trial partial "
                        "response exceeds routine primary care management [53, 55]."
                    )
                    out.non_med_recommendations.append(
                        "Reassess before escalating: diagnostic accuracy, medication adherence, "
                        "comorbidities contributing to treatment resistance [45, 53, 54]."
                    )
                    out.non_med_recommendations.append(
                        "Advanced interventions to discuss with psychiatry: electroconvulsive "
                        "therapy (ECT), transcranial magnetic stimulation (TMS), ketamine, "
                        "esketamine [45, 53, 54]."
                    )
                    out.warnings.append(
                        "STAR*D note: 67% cumulative remission across four treatment levels; "
                        "no definitive evidence base for partial response management after "
                        "multiple trials [25]."
                    )
                    out.rationale.append(
                        "Step 6b: trial 3 partial response — psychiatric consultation advisable; "
                        "advanced interventions flagged [25, 45, 53, 54, 55]."
                    )
                    aug_recs = build_augmentation_recs(
                        p, current_med, pathways, can_recommend, state
                    )
                    out.recommendations = [continue_reco] + aug_recs
                    out.audit_trail.append("branch:step_6b_trial3_partial")
                    return state

                elif trial_n == 2:
                    # Step 6a: second-trial partial response
                    out.non_med_recommendations.append(
                        "Reassess before escalating: diagnostic accuracy, medication adherence, "
                        "comorbidities contributing to treatment resistance [45, 53, 54]."
                    )
                    out.rationale.append(
                        "Step 6a: trial 2 partial response — tolerability determines next "
                        "step [45, 53, 54]."
                    )
                    if p.tolerability == "good":
                        out.rationale.append(
                            "Good tolerability: augmentation preferred before advancing to "
                            "trial 3 [45, 53]."
                        )
                        aug_recs = build_augmentation_recs(
                            p, current_med, pathways, can_recommend, state
                        )
                        out.recommendations = [continue_reco] + aug_recs
                        out.audit_trail.append("branch:step_6a_good_tolerability_augment")
                    else:
                        out.rationale.append(
                            "Poor tolerability: switch to trial 3 medication "
                            "(untried class) [25, 55]."
                        )
                        pool = third_line_pool()
                        picks = pick_two(pool)
                        out.recommendations = [
                            _make_reco(
                                p, m, intent="switch_to", active_paths=pathways,
                                additional_messages=[
                                    _switch_taper_message(current_med, m)
                                ],
                                rationale=[
                                    "Step 6a — poor tolerability: switch to trial 3 "
                                    "untried class [25, 55]."
                                ],
                                evidence=[
                                    EvidenceItem(variable="Trial number", value="2", fhir_source="Derived"),
                                    EvidenceItem(variable="Tolerability", value="poor", fhir_source="Derived"),
                                ],
                                cap_override=max_cap_for_context(p, m, pathways),
                            )
                            for m in picks
                        ]
                        if not out.recommendations:
                            out.warnings.append("No safe switch option remains; clinician review.")
                        out.audit_trail.append("branch:step_6a_poor_tolerability_switch")
                    return state

                else:
                    # Trial 1 (or unspecified) — standard Step 6 augmentation hierarchy
                    out.rationale.append(
                        "Step 6: partial response at max dose → three-tier augmentation "
                        "hierarchy [1, 25, 36, 37, 38, 39, 40, 41, 45, 47, 48]."
                    )
                    aug_recs = build_augmentation_recs(
                        p, current_med, pathways, can_recommend, state
                    )
                    out.recommendations = [continue_reco] + aug_recs
                    out.audit_trail.append("branch:step_6_trial1_augment")
                    return state

            if resp_cat == "no_response":
                if trial_n == 1:
                    pool, why = second_line_pool(), "No response after Trial 1: switch to Trial 2."
                elif trial_n == 2:
                    pool, why = third_line_pool(), "No response after Trial 2: switch to Trial 3 (untried class)."
                else:
                    pool = third_line_pool()
                    why = "No response after Trial 3: psychiatric consultation and advanced strategies indicated."
                picks = pick_two(pool)
                out.recommendations = [
                    _make_reco(
                        p, m, intent="switch_to", active_paths=pathways,
                        additional_messages=[_switch_taper_message(current_med, m)],
                        rationale=[why],
                        evidence=[
                            EvidenceItem(variable="Trial number", value=str(trial_n), fhir_source="Derived"),
                            EvidenceItem(variable="Response", value="no_response", fhir_source="Derived"),
                        ],
                        cap_override=max_cap_for_context(p, m, pathways),
                    )
                    for m in picks
                ]
                if not out.recommendations:
                    out.warnings.append("No safe switch option remains; clinician review.")
                out.rationale.append(why)
                return state

        # --------------------------------------------------------
        # New start
        # --------------------------------------------------------
        if p.phq_current < 10:
            out.non_med_recommendations.append("PHQ < 10 without follow-up context: no medication recommended.")
            out.rationale.append("PHQ < 10, no valid follow-up context.")
            return state

        if trial_n == 1:
            pool, why = first_line_pool(), "Trial 1: first-line selection per all active path(s)."
        elif trial_n == 2:
            pool, why = second_line_pool(), "Trial 2: alternate antidepressant per history/tolerability."
        else:
            pool, why = third_line_pool(), "Trial 3: untried medication class preferred."

        picks = pick_two(pool)
        # Fallback if fewer than 2
        if len(picks) < 2:
            picks_set = set(picks)
            for m in MED_KB:
                if len(picks) >= 2:
                    break
                if m not in picks_set and can_recommend(m):
                    picks.append(m)

        out.recommendations = [
            _make_reco(
                p, m, intent="start", active_paths=pathways,
                additional_messages=["Initiate; titrate every 1–2 weeks; reassess at 6 weeks for response. [3, 4]"],
                rationale=[why],
                evidence=[
                    EvidenceItem(variable="Trial number", value=str(trial_n), fhir_source="Derived"),
                    EvidenceItem(variable="PHQ-9 current", value=str(p.phq_current), fhir_source="Observation"),
                    EvidenceItem(variable="Active paths", value=str(pathways), fhir_source="Derived"),
                ],
                cap_override=max_cap_for_context(p, m, pathways),
            )
            for m in picks[:2]
        ]
        if not out.recommendations:
            out.warnings.append("All candidates blocked; clinician review required.")
            out.rationale.append("All options blocked or excluded by history.")
            return state

        # Surface path-adjustment rationale
        notes: List[str] = []
        if "geriatric" in pathways:
            notes.append("geriatric: paroxetine/fluoxetine excluded; escitalopram capped at 10 mg, citalopram at 20 mg")
        if "cardiac" in pathways:
            notes.append("cardiac: sertraline/fluoxetine preferred; QTc-conditional exclusions active if QTc > 500 ms")
        if "renal" in pathways:
            renal_val = p.crcl_ml_min if p.crcl_ml_min is not None else p.egfr_ml_min_1_73
            bucket = renal_bucket_from_crcl(p.crcl_ml_min, p.egfr_ml_min_1_73)
            notes.append(f"renal (eGFR/CrCl {renal_val} → bucket {bucket}): dose caps applied per RENAL_MAX_CAPS_MG")
        if "hepatic" in pathways:
            notes.append("hepatic: duloxetine excluded; SSRI doses reduced")
        if "pain" in pathways:
            notes.append("pain: SNRIs anchored at front of pool")
        if "insomnia" in pathways and "obesity" not in pathways:
            notes.append("insomnia: mirtazapine included in pool")
        if "insomnia" in pathways and "obesity" in pathways:
            notes.append("insomnia + obesity conflict: mirtazapine excluded (obesity path wins)")
        if notes:
            out.rationale.append("Active path adjustments applied: " + " | ".join(notes))
        out.rationale.append(why)
        return state


def _discontinuation_taper_note(p: PatientInput) -> str:
    """Return a medication-specific taper note for the maintenance plan."""
    med = p.current_antidepressant_key
    if med and med in EXTENDED_TAPER_MEDS:
        disp = MED_KB.get(med, {}).get("display", med)
        return (
            f"Tapering — {disp}: EXTENDED taper required (high discontinuation syndrome "
            f"risk). Reduce by 25% every 2 weeks; hold at lowest available dose ≥4 weeks "
            f"before stopping. Consider hyperbolic taper for long-term users. "
            f"[87, 88, 93, 94]"
        )
    return (
        "Taper slowly on discontinuation — recommend MoodCalmer dCBT alongside tapering "
        "to support psychological adjustment. [25, 96, 97]"
    )


class OutputCompletionStage(Stage):
    """
    Populates the five structured output sections (Round 5 of CLAUDE_MDD.md spec):
      1. monitoring_schedule  — standard monitoring table + path-specific additions
      2. maintenance_plan     — based on prior_depressive_episodes
      3. medications_excluded — surfaced from state.exclusion_reasons
      4. adjunctive_options   — Tier 4: omega-3 + celecoxib (with contraindication checks)
      5. reference_list       — all [Xn] citations found in output text, resolved from REFERENCE_DB
    """
    name = "output_completion"

    def run(self, state: WorkingState) -> WorkingState:
        import re as _re
        p   = state.patient
        out = state.output

        # 0. Serotonin syndrome burden check ─────────────────────────────────
        _sero_count = 0
        _sero_sources: List[str] = []
        _pm_norm = {m.lower().strip() for m in (p.current_medications or [])}
        # Check current antidepressant
        if p.current_antidepressant_key and p.current_antidepressant_key in SEROTONERGIC_AD_KEYS:
            _sero_count += 1
            _sero_sources.append(p.current_antidepressant_key)
        # Check existing co-medications for serotonergic agents
        _SERO_COMED_TERMS = {
            "tramadol", "fentanyl", "meperidine", "methadone", "tapentadol",
            "linezolid", "methylene blue", "dextromethorphan", "triptans",
            "sumatriptan", "lithium", "buspirone",
        }
        _OPIOID_SERO_TERMS = {"tramadol", "meperidine", "tapentadol", "fentanyl"}
        _current_is_ssri_snri = (
            p.current_antidepressant_key and
            p.current_antidepressant_key in (SSRI_KEYS | SNRI_KEYS)
        )
        for cm in _pm_norm:
            if any(t in cm for t in _SERO_COMED_TERMS):
                _sero_count += 1
                _sero_sources.append(f"co-med: {cm}")
            # Special flag: tramadol/serotonergic opioid + SSRI/SNRI combination
            if any(t in cm for t in _OPIOID_SERO_TERMS) and _current_is_ssri_snri:
                out.warnings.append(
                    f"SEROTONIN SYNDROME RISK — {cm} + {p.current_antidepressant_key}: "
                    f"serotonergic opioid co-prescribed with SSRI/SNRI. "
                    f"This is the most common pharmacodynamic serotonin syndrome "
                    f"interaction. Review necessity or consider opioid substitution. "
                    f"[89, 90, 91]"
                )
        # Augmentation only: serotonergic agent added on top of a current serotonergic AD
        # (start/switch_to are alternatives, not concurrent — never counted here)
        if p.current_antidepressant_key in SEROTONERGIC_AD_KEYS:
            for _r in out.recommendations:
                if _r.medication_key in SEROTONERGIC_AD_KEYS and _r.intent == "augment_with":
                    _sero_count += 1
                    _sero_sources.append(f"augment: {_r.medication_key}")
        if _sero_count >= 2:
            out.warnings.append(
                f"SEROTONIN SYNDROME RISK: {_sero_count} serotonergic agents identified "
                f"({'; '.join(_sero_sources)}). Avoid concurrent serotonergic agents unless "
                f"benefit clearly outweighs risk. Monitor for agitation, diaphoresis, tremor, "
                f"hyperreflexia, clonus, hyperthermia. If suspected: discontinue all "
                f"serotonergic agents and seek emergency evaluation. [89, 90, 91, 98]"
            )
            out.audit_trail.append(f"serotonin_syndrome_risk:count={_sero_count}")

        # 0b. Switching protocol — class-to-class lookup ──────────────────────
        _switch_recs = [_r for _r in out.recommendations if _r.intent == "switch_to"]
        if _switch_recs:
            _outgoing = p.current_antidepressant_key
            _seen_protocols: set = set()  # deduplicate identical (prior, new_cls) pairs
            for _srec in _switch_recs:
                _new_key = _srec.medication_key
                _proto   = get_switching_protocol(_outgoing, _new_key)
                _pair_key = (classify_med_for_switch(_outgoing),
                             classify_med_for_switch(_new_key))
                if _pair_key in _seen_protocols:
                    continue
                _seen_protocols.add(_pair_key)
                # Format and append protocol lines
                _lines = _format_switch_protocol_lines(_outgoing, _new_key, _proto)
                out.switching_protocol.extend(_lines)
                out.switching_protocol.append("")  # blank separator between protocols
                # MAOI washout safety flag
                if _proto.method == "washout":
                    out.warnings.append(
                        f"WASHOUT REQUIRED before starting "
                        f"{MED_KB.get(_new_key, {}).get('display', _new_key)}: "
                        f"{_proto.duration}. {_proto.warning} "
                        f"[{chr(44).join(_proto.citations)}]"
                    )
                # Cross-taper overlap of two serotonergic agents
                if (_proto.method == "cross_taper"
                        and _outgoing in SEROTONERGIC_AD_KEYS
                        and _new_key in SEROTONERGIC_AD_KEYS):
                    out.warnings.append(
                        f"SEROTONIN SYNDROME RISK — cross-taper overlap: "
                        f"{MED_KB.get(_outgoing, {}).get('display', _outgoing)} → "
                        f"{MED_KB.get(_new_key, {}).get('display', _new_key)}. "
                        f"Both agents are serotonergic. Monitor closely during overlap period "
                        f"for agitation, diaphoresis, tremor, hyperreflexia, clonus, "
                        f"hyperthermia. Consider sequential approach if feasible. [89, 90, 91, 98]"
                    )
                    out.audit_trail.append(
                        f"serotonin_syndrome_risk:cross_taper:{_outgoing}->{_new_key}"
                    )
                out.audit_trail.append(
                    f"switching_protocol:{classify_med_for_switch(_outgoing)}"                    f"->{classify_med_for_switch(_new_key)}:method={_proto.method}"
                )
            # Trim trailing blank separator
            while out.switching_protocol and out.switching_protocol[-1] == "":
                out.switching_protocol.pop()
            # FINISH mnemonic when extended taper meds are in the outgoing position
            if _outgoing and _outgoing in EXTENDED_TAPER_MEDS:
                out.switching_protocol.append(
                    "Patient education — FINISH mnemonic: Flu-like symptoms, Insomnia, "
                    "Nausea, Imbalance, Sensory disturbances, Hyperarousal. [28]"
                )

        # 1. Monitoring schedule ────────────────────────────────────────────
        out.monitoring_schedule = [
            "Initiation: baseline weight, BMI, BP. [3]",
            "Week 2: assess tolerability and adherence. [26]",
            "Week 4–6: dose adjustment if inadequate response. [26]",
            "Week 8–12: full response assessment; switch if no response. [26]",
            "Weeks 1–4: close suicidality monitoring — all patients; especially under 25. [73, 74]",
        ]
        if any(r.medication_class == "SGA" for r in out.recommendations
               if r.intent == "augment_with"):
            out.monitoring_schedule.append(
                "Antipsychotic augmentation: baseline lipids and HbA1c; BMI monthly ×3 then "
                "quarterly; BP, lipids, HbA1c at 3 months then annually. [3]"
            )

        # 2. Maintenance plan ───────────────────────────────────────────────
        episodes = p.prior_depressive_episodes
        if episodes is None:
            out.maintenance_plan = [
                "Episode history unknown: discuss maintenance duration with patient. "
                "[2, 4, 18, 70, 71, 72]",
                _discontinuation_taper_note(p),
            ]
        elif episodes == 0:
            _taper_note_0 = _discontinuation_taper_note(p)
            out.maintenance_plan = [
                "First episode: continue medication 6–9 months after remission. [2, 4, 71]",
                _taper_note_0,
            ]
        elif episodes == 1:
            _taper_note_1 = _discontinuation_taper_note(p)
            out.maintenance_plan = [
                "Recurrent depression (1 prior episode): continue at least 2 years. [18, 70, 71, 72]",
                "Discuss long-term risk; consider extending if high relapse risk. [2, 4]",
                _taper_note_1,
            ]
        else:
            out.maintenance_plan = [
                f"Recurrent depression ({episodes} prior episodes): consider indefinite "
                "continuation due to high relapse risk. [2, 4, 18, 70, 71, 72]",
                _discontinuation_taper_note(p),
            ]

        # 3. Medications excluded ───────────────────────────────────────────
        if state.exclusion_reasons:
            out.medications_excluded = [
                f"{MED_KB.get(med, {}).get('display', med)}: {reason}"
                for med, reason in sorted(state.exclusion_reasons.items())
            ]
        else:
            out.medications_excluded = [
                "No medications excluded by deviation path rules for this patient context."
            ]

        # 4. Adjunctive options — Tier 4 ──────────────────────────────────────
        out.adjunctive_options.append(
            "Omega-3 fatty acids (EPA ≥60%, 1–1.5 g/day): SOR B. Small benefit "
            "(SMD −0.40), very low certainty; VA/DoD found insufficient evidence to recommend "
            "over validated medications. May consider adding. "
            "[4, 64, 65, 66, 67, 68]"
        )
        renal_val = p.crcl_ml_min if p.crcl_ml_min is not None else p.egfr_ml_min_1_73
        celecoxib_ci: List[str] = []
        if p.cardiac_history:
            celecoxib_ci.append("cardiac history")
        if renal_val is not None and renal_val < 60:
            celecoxib_ci.append(f"eGFR/CrCl {renal_val} < 60")
        if celecoxib_ci:
            out.adjunctive_options.append(
                f"Celecoxib 400 mg/day ×6 weeks: CONTRAINDICATED in this patient "
                f"({'; '.join(celecoxib_ci)}). Do not prescribe. [60, 61, 63]"
            )
        else:
            out.adjunctive_options.append(
                "Celecoxib 400 mg/day ×6 weeks: SOR B. Moderate benefit (SMD −0.82, "
                "OR for remission 7.89) but high risk of bias. Confirm no cardiac history, "
                "eGFR ≥60, no NSAID contraindications before prescribing. May consider adding. "
                "[60, 61, 63]"
            )

        # 5. Reference list ─────────────────────────────────────────────────
        all_text = " ".join([
            *out.findings, *out.rationale, *out.warnings,
            *out.non_med_recommendations, *out.monitoring_schedule,
            *out.maintenance_plan, *out.adjunctive_options,
            *[m for r in out.recommendations for m in r.messages + r.rationale],
        ])
        cited: Set[str] = set()
        for _m in _re.finditer(r'\[(\d+(?:[,\s]+\d+)*)\]', all_text):
            for _num in _re.split(r'[,\s]+', _m.group(1)):
                _num = _num.strip()
                if _num:
                    cited.add(_num)
        # Sort by plain consolidated number
        def _sort_key(k: str) -> int:
            try:
                return int(k)
            except (ValueError, TypeError):
                return 9999
        out.reference_list = [
            REFERENCE_DB[k]
            for k in sorted(cited, key=_sort_key)
            if k in REFERENCE_DB
        ]
        # Always include monitoring anchor refs
        for k in ["26", "73", "74", "3", "4"]:
            entry = REFERENCE_DB.get(k)
            if entry and entry not in out.reference_list:
                out.reference_list.append(entry)

        return state



# ============================================================
# Report formatter helpers (section 9 — final output format)
# ============================================================
_CITE_RE = re.compile(r'\s*\[\d+(?:[,\s]+\d+)*\]')


def _phq_tier(phq: int) -> str:
    if phq < 5:  return "minimal"
    if phq < 10: return "mild"
    if phq < 15: return "moderate"
    if phq < 20: return "moderate-severe"
    return "severe"


def _phq_label(phq: int) -> str:
    tier = _phq_tier(phq)
    ranges = {
        "minimal":        "0\u20134",
        "mild":           "5\u20139",
        "moderate":       "10\u201314",
        "moderate-severe":"15\u201319",
        "severe":         "20\u201327",
    }
    return f"{tier.title()} (PHQ {ranges[tier]})"


def _strip_cites(text: str) -> str:
    text = _CITE_RE.sub("", text)
    # Remove comma/semicolon debris left between adjacent stripped brackets
    text = re.sub(r'(?<=[.,])\s*,+', '', text)
    text = re.sub(r',\s*$', '', text.rstrip())
    return text.strip()


def _build_patient_summary(p: "PatientInput", out: "AlgorithmOutput") -> dict:
    comorbidities: List[str] = []
    if p.insomnia:             comorbidities.append("insomnia")
    if p.fibromyalgia:         comorbidities.append("fibromyalgia/chronic pain")
    if p.anxiety_comorbidity:  comorbidities.append("anxiety")
    if p.cardiac_history:      comorbidities.append("cardiac history")
    if p.hepatic_impairment:   comorbidities.append("hepatic impairment")
    if p.pregnant:             comorbidities.append("pregnant")
    if p.dementia:             comorbidities.append("dementia")

    flags: List[str] = []
    if p.suicidality != "none":                        flags.append(f"suicidality={p.suicidality}")
    if p.chronic_suicidality:                          flags.append("chronic suicidality")
    if p.qtc_ms is not None and p.qtc_ms > 450:        flags.append(f"QTc={p.qtc_ms}ms")

    summary: dict = {
        "age": p.age,
        "phq_score": p.phq_current,
        "severity": _phq_label(p.phq_current),
        "trial_number": p.trial_number or 1,
    }
    if p.current_antidepressant_key:
        summary["current_medication"] = p.current_antidepressant_key
        summary["weeks_on_treatment"] = p.weeks_on_current_antidepressant
    if p.baseline_phq is not None and p.baseline_phq > 0:
        pct = round((p.baseline_phq - p.phq_current) / p.baseline_phq * 100)
        summary["phq_improvement_pct"] = pct
    if comorbidities:
        summary["comorbidities"] = comorbidities
    if flags:
        summary["notable_flags"] = flags
    if p.therapy_preference:
        summary["therapy_preference"] = p.therapy_preference
    return summary


def _build_safety_flags(out: "AlgorithmOutput") -> List[str]:
    return [_strip_cites(w) for w in out.warnings]


def _build_active_pathways(out: "AlgorithmOutput") -> List[str]:
    default = {"adult"}
    return [p for p in out.pathway_applied if p not in default]


def _build_medication_entries(p: "PatientInput", out: "AlgorithmOutput") -> List[dict]:
    entries: List[dict] = []
    for rec in out.recommendations:
        if rec.intent not in ("start", "continue", "switch_to", "increase"):
            continue
        entry: dict = {
            "medication": rec.medication_display,
            "class": rec.medication_class,
            "intent": rec.intent,
        }
        if rec.intent == "start":
            entry["dose"] = f"Start: {rec.start_dose} \u2192 Target: {rec.dose_range}"
        elif rec.intent == "switch_to":
            entry["dose"] = f"Target: {rec.dose_range}"
        else:  # continue / increase
            dose_curr = f"{p.current_dose_mg} mg" if p.current_dose_mg else "current"
            entry["dose"] = f"Current: {dose_curr} \u2192 Target: {rec.dose_range}"
        if rec.messages:
            entry["notes"] = [_strip_cites(m) for m in rec.messages]
        entries.append(entry)
    return entries


def _build_therapy_rec(p: "PatientInput", out: "AlgorithmOutput") -> Optional[dict]:
    tier = _phq_tier(p.phq_current)
    if tier == "minimal":
        return None
    if tier == "mild":
        return {
            "level": "self-directed digital CBT",
            "recommendation": "MoodCalmer dCBT \u2014 self-guided internet-based CBT program",
            "format": "digital self-directed",
            "references": ["99", "101"],
        }
    elif tier == "moderate":
        return {
            "level": "shared decision",
            "recommendation": (
                "Psychotherapy (CBT, IPT) alongside medication or as first-line; "
                "share decision with patient"
            ),
            "format": "individual face-to-face or structured digital",
            "references": ["1", "2", "3", "4", "100"],
        }
    else:  # moderate-severe / severe
        return {
            "level": "recommended combination",
            "recommendation": (
                "Face-to-face psychotherapy (CBT or IPT) combined with medication \u2014 "
                "combination superior to either alone at PHQ \u226515"
            ),
            "format": "individual face-to-face psychotherapy",
            "references": ["1", "3", "4", "100"],
        }


def _build_switch_entries(out: "AlgorithmOutput") -> List[dict]:
    if not out.switching_protocol:
        return []
    entries: List[dict] = []
    current: dict = {}
    for line in out.switching_protocol:
        line = line.strip()
        if not line:
            if current:
                entries.append(current)
                current = {}
            continue
        if line.startswith("Prior medication:"):
            current["prior"] = line.replace("Prior medication:", "").strip()
        elif line.startswith("New medication:"):
            current["new"] = line.replace("New medication:", "").strip()
        elif line.startswith("Method:"):
            current["method"] = line.replace("Method:", "").strip()
        elif line.startswith("Duration:"):
            current["duration"] = line.replace("Duration:", "").strip()
        elif line.startswith("Warning:"):
            current["warning"] = _strip_cites(line.replace("Warning:", "").strip())
        elif line.startswith("Safety flag:") or line.startswith("\u2139"):
            current.setdefault("safety_flags", []).append(_strip_cites(line))
        elif "FINISH" in line:
            current.setdefault("mnemonics", []).append(_strip_cites(line))
    if current:
        entries.append(current)
    return entries


def _build_meds_excluded(out: "AlgorithmOutput") -> List[str]:
    return [
        _strip_cites(m) for m in out.medications_excluded
        if not m.lower().startswith("no medications excluded by deviation path")
    ]


def _build_augmentation_entries(out: "AlgorithmOutput") -> List[dict]:
    entries: List[dict] = []
    for rec in out.recommendations:
        if rec.intent != "augment_with":
            continue
        entry: dict = {
            "medication": rec.medication_display,
            "class": rec.medication_class,
            "dose": f"Target: {rec.dose_range}",
        }
        if rec.rationale:
            entry["rationale"] = _strip_cites(rec.rationale[0])
        entries.append(entry)
    return entries


def _build_next_check_in(p: "PatientInput", out: "AlgorithmOutput") -> str:
    has_switch = any(r.intent == "switch_to" for r in out.recommendations)
    has_start  = any(r.intent == "start"     for r in out.recommendations)
    if has_switch or has_start:
        return "Week 2: tolerability check and adherence assessment"
    if p.phq_current <= 9:
        return "Week 8\u201312: reassess PHQ-9 and response to treatment"
    weeks = p.weeks_on_current_antidepressant or 0
    if weeks < 6:
        return "Week 4\u20136: dose adjustment assessment"
    return "Week 8\u201312: full response assessment"


def _collect_citations_from(*text_lists: List[str]) -> List[str]:
    pat = re.compile(r'\[(\d+(?:[,\s]+\d+)*)\]')
    nums: Set[str] = set()
    for texts in text_lists:
        for t in (texts or []):
            for m in pat.finditer(t):
                for n in re.split(r'[,\s]+', m.group(1)):
                    if n.strip():
                        nums.add(n.strip())
    return sorted(nums, key=lambda x: int(x))


def _build_references(cite_nums: List[str]) -> dict:
    return {n: REFERENCE_DB[n] for n in cite_nums if n in REFERENCE_DB}


def _format_text_report(report: dict) -> str:
    SEP  = "\u2500" * 70
    SEP2 = "\u2550" * 70
    lines: List[str] = []

    lines.append(SEP2)
    lines.append("  MDD CLINICAL DECISION SUPPORT \u2014 REPORT")
    lines.append(SEP2)

    ps = report.get("patient_summary", {})
    lines.append(
        f"\n  Patient: age {ps.get('age')}, "
        f"PHQ-9 = {ps.get('phq_score')} ({ps.get('severity', '')})"
    )
    lines.append(f"  Trial #{ps.get('trial_number', 1)}")
    if ps.get("current_medication"):
        lines.append(f"  Current: {ps['current_medication']}, {ps.get('weeks_on_treatment', '?')} weeks")
        if ps.get("phq_improvement_pct") is not None:
            lines.append(f"  PHQ improvement: {ps['phq_improvement_pct']}%")
    if ps.get("comorbidities"):
        lines.append(f"  Comorbidities: {', '.join(ps['comorbidities'])}")

    if report.get("safety_flags"):
        lines.append(f"\n{SEP}")
        lines.append("  SAFETY FLAGS")
        lines.append(SEP)
        for flag in report["safety_flags"]:
            lines.append(f"  \u26a0  {flag}")

    if report.get("active_pathways"):
        lines.append(f"\n{SEP}")
        lines.append("  ACTIVE PATHWAYS")
        lines.append(SEP)
        for path in report["active_pathways"]:
            lines.append(f"  \u2022 {path}")

    recs   = report.get("recommendations", {})
    meds   = recs.get("medications", [])
    therapy = recs.get("therapy")
    if meds or therapy:
        lines.append(f"\n{SEP}")
        lines.append("  RECOMMENDATIONS")
        lines.append(SEP)
        if meds:
            lines.append("  Medications:")
            for i, m in enumerate(meds, 1):
                lines.append(f"    [{i}] {m['medication']}  ({m['intent']})")
                lines.append(f"        Dose: {m.get('dose', '')}")
                for note in m.get("notes", []):
                    lines.append(f"        \u2192 {note}")
        if therapy:
            lines.append("  Therapy:")
            lines.append(f"    Level: {therapy.get('level', '').title()}")
            lines.append(f"    {therapy.get('recommendation', '')}")
            lines.append(f"    Format: {therapy.get('format', '').title()}")

    if report.get("switching_protocol"):
        lines.append(f"\n{SEP}")
        lines.append("  SWITCHING PROTOCOL")
        lines.append(SEP)
        for entry in report["switching_protocol"]:
            if entry.get("prior"):
                lines.append(f"  Prior: {entry['prior']}")
            if entry.get("new"):
                lines.append(f"  New:   {entry['new']}")
            if entry.get("method"):
                lines.append(f"  Method: {entry['method']}  |  Duration: {entry.get('duration', 'N/A')}")
            if entry.get("warning"):
                lines.append(f"  \u26a0  {entry['warning']}")
            for f in entry.get("safety_flags", []):
                lines.append(f"  \u2139  {f}")
            for mn in entry.get("mnemonics", []):
                lines.append(f"  \u2139  {mn}")

    if report.get("medications_excluded"):
        lines.append(f"\n{SEP}")
        lines.append("  MEDICATIONS EXCLUDED")
        lines.append(SEP)
        for exc in report["medications_excluded"]:
            lines.append(f"  \u2717  {exc}")

    if report.get("augmentation_plan"):
        lines.append(f"\n{SEP}")
        lines.append("  AUGMENTATION PLAN")
        lines.append(SEP)
        for i, aug in enumerate(report["augmentation_plan"], 1):
            lines.append(f"  [{i}] {aug['medication']}")
            lines.append(f"      Dose: {aug.get('dose', '')}")
            if aug.get("rationale"):
                lines.append(f"      {aug['rationale']}")

    if report.get("next_check_in"):
        lines.append(f"\n{SEP}")
        lines.append("  NEXT CHECK-IN")
        lines.append(SEP)
        lines.append(f"  {report['next_check_in']}")

    if report.get("references"):
        lines.append(f"\n{SEP}")
        lines.append("  REFERENCES")
        lines.append(SEP)
        for n, ref in sorted(report["references"].items(), key=lambda x: int(x[0])):
            lines.append(f"  {ref}")

    lines.append(f"\n{SEP2}")
    return "\n".join(lines)


class ReportFormatterStage(Stage):
    name = "report_formatter"

    def run(self, state: "WorkingState") -> "WorkingState":
        p   = state.patient
        out = state.output

        all_texts: List[str] = (
            out.findings
            + out.rationale
            + out.warnings
            + out.maintenance_plan
            + out.monitoring_schedule
            + out.switching_protocol
            + out.adjunctive_options
            + [m for rec in out.recommendations for m in rec.messages]
            + [m for rec in out.recommendations for m in rec.rationale]
            + out.non_med_recommendations
        )

        patient_summary  = _build_patient_summary(p, out)
        safety_flags     = _build_safety_flags(out)
        active_pathways  = _build_active_pathways(out)
        medications      = _build_medication_entries(p, out)
        therapy          = _build_therapy_rec(p, out)
        switch_entries   = _build_switch_entries(out)
        meds_excluded    = _build_meds_excluded(out)
        augmentation     = _build_augmentation_entries(out)
        next_check_in    = _build_next_check_in(p, out)

        cite_nums = _collect_citations_from(all_texts)
        if therapy:
            cite_nums = sorted(
                set(cite_nums) | set(therapy.get("references", [])),
                key=lambda x: int(x),
            )
        references = _build_references(cite_nums)

        report: dict = {"patient_summary": patient_summary}
        if safety_flags:
            report["safety_flags"] = safety_flags
        if active_pathways:
            report["active_pathways"] = active_pathways
        recs_section: dict = {}
        if medications:
            recs_section["medications"] = medications
        if therapy:
            recs_section["therapy"] = therapy
        if recs_section:
            report["recommendations"] = recs_section
        if switch_entries:
            report["switching_protocol"] = switch_entries
        if meds_excluded:
            report["medications_excluded"] = meds_excluded
        if augmentation:
            report["augmentation_plan"] = augmentation
        report["next_check_in"] = next_check_in
        report["references"] = references

        out.report      = report
        out.text_report = _format_text_report(report)
        return state


class FinalizeStage(Stage):
    name = "finalize"

    def run(self, state: WorkingState) -> WorkingState:
        out = state.output
        seen: Set[str] = set()
        deduped: List[str] = []
        for w in out.warnings:
            if w not in seen:
                deduped.append(w)
                seen.add(w)
        out.warnings = deduped
        if not out.recommendations and not out.non_med_recommendations and not out.stop_reason:
            out.non_med_recommendations.append("No recommendation generated — clinician review required.")
            out.rationale.append("Insufficient data or all options blocked.")
        return state


# ------------------------------------------------------------
# 11) Engine
# ------------------------------------------------------------
class MDDEngine:
    """
    MDD algorithm engine with multi-path routing (Step 2 of CLAUDE_MDD.md).

    detect_active_paths() fires ALL applicable deviation paths simultaneously.
    Each path contributes exclusions, dose caps, and preferred-medication
    ordering. Cross-path conflicts are surfaced in output.path_conflicts
    rather than silently resolved.

    Pipeline:
      findings → pathway (multi) → mania exclusion → QTc safety →
      severity/non-med → med safety → clinical contraindications
      (all-path exclusions + conflict detection) → dose sanity →
      treatment selection (multi-path pool) → finalize
    """
    def __init__(self) -> None:
        self.stages: List[Stage] = [
            FindingsStage(),
            PathwayStage(),
            SuicidalityScreenStage(),
            ManiaExclusionStage(),
            QTcSafetyStage(),
            SeverityAndNonMedStage(),
            MedSafetyStage(),
            ClinicalContraindicationStage(),
            DoseSanityStage(),
            TreatmentSelectionStage(),
            OutputCompletionStage(), ReportFormatterStage(),
            FinalizeStage(),
        ]

    def run(self, patient: PatientInput) -> AlgorithmOutput:
        state = WorkingState(
            patient=patient,
            output=AlgorithmOutput(),
            blocked_candidates=set(),
        )
        for stage in self.stages:
            state.output.audit_trail.append(f"start:{stage.name}")
            state = stage.run(state)
            state.output.audit_trail.append(f"end:{stage.name}")
        return state.output


engine = MDDEngine()
print("MDDEngine ready. Use: engine.run(PatientInput(...))")


In [ ]:
# ============================================================
# Test: Suicidality screen — three cases, all age 22
# Expected:
#   Case 1 (acutely_suicidal)   → hard stop, emergent hospitalization, no meds
#   Case 2 (elevated_ideation)  → warning + monitoring, algorithm continues
#   Case 3 (none)               → normal flow
#   All three                   → FDA black box warning (age < 25)
# ============================================================

def show_result(label: str, result) -> None:
    SEP = "─" * 62
    print(f"\n{'═'*62}")
    print(f"  {label}")
    print(f"{'═'*62}")

    print(f"\n  STOP REASON : {result.stop_reason or '(none — algorithm ran to completion)'}")
    print(f"  ACTIVE PATHS: {result.pathway_applied}")

    print(f"\n{SEP}")
    print("  WARNINGS")
    print(SEP)
    for w in result.warnings:
        marker = "🔴" if "black box" in w.lower() or "acutely" in w.lower() else "⚠ "
        print(f"  {marker}  {w}")

    print(f"\n{SEP}")
    print("  NON-MEDICATION RECOMMENDATIONS")
    print(SEP)
    for nm in result.non_med_recommendations:
        print(f"  •  {nm}")

    print(f"\n{SEP}")
    print("  MEDICATION RECOMMENDATIONS")
    print(SEP)
    if result.recommendations:
        for r in result.recommendations:
            print(f"  [{r.intent.upper()}]  {r.medication_display}  |  start: {r.start_dose}")
    else:
        print("  (none generated)")


# ── Case 1: acutely suicidal ──────────────────────────────────────────────────
case1 = engine.run(PatientInput(
    age=22, phq_current=18, mania_screen="negative",
    suicidality="acutely_suicidal",
))
show_result("CASE 1 — Age 22, suicidality: acutely_suicidal", case1)

# ── Case 2: elevated ideation ─────────────────────────────────────────────────
case2 = engine.run(PatientInput(
    age=22, phq_current=18, mania_screen="negative",
    suicidality="elevated_ideation",
))
show_result("CASE 2 — Age 22, suicidality: elevated_ideation", case2)

# ── Case 3: none ──────────────────────────────────────────────────────────────
case3 = engine.run(PatientInput(
    age=22, phq_current=18, mania_screen="negative",
    suicidality="none",
))
show_result("CASE 3 — Age 22, suicidality: none", case3)

# ── Summary assertion ─────────────────────────────────────────────────────────
BLACK_BOX_TEXT = "FDA black box warning"
print(f"\n{'═'*62}")
print("  SUMMARY CHECKS")
print(f"{'═'*62}")

checks = [
    ("Case 1: stop_reason == acutely_suicidal",
     case1.stop_reason == "acutely_suicidal"),
    ("Case 1: no medication recommendations",
     len(case1.recommendations) == 0),
    ("Case 1: emergent hospitalization in non_med_recs",
     any("EMERGENT" in r for r in case1.non_med_recommendations)),
    ("Case 2: no stop_reason",
     case2.stop_reason is None),
    ("Case 2: medication recommendations generated",
     len(case2.recommendations) > 0),
    ("Case 2: hospitalization consideration in warnings",
     any("hospitalisation" in w or "hospitalization" in w for w in case2.warnings)),
    ("Case 3: no stop_reason",
     case3.stop_reason is None),
    ("Case 3: medication recommendations generated",
     len(case3.recommendations) > 0),
    ("Case 1: FDA black box warning present",
     any(BLACK_BOX_TEXT in w for w in case1.warnings)),
    ("Case 2: FDA black box warning present",
     any(BLACK_BOX_TEXT in w for w in case2.warnings)),
    ("Case 3: FDA black box warning present",
     any(BLACK_BOX_TEXT in w for w in case3.warnings)),
]

all_passed = True
for label, passed in checks:
    status = "PASS" if passed else "FAIL"
    print(f"  [{status}]  {label}")
    if not passed:
        all_passed = False

print(f"\n  {'All checks passed.' if all_passed else 'FAILURES — review above.'}")


In [ ]:
# ============================================================
# Test: Dose corrections per Step 4 of CLAUDE_MDD.md
#
# Case 1: Age 68, hepatic impairment, escitalopram
#   → Active paths: geriatric + hepatic
#   → Cap: 10 mg (min of geriatric 10mg and hepatic 10mg)
#   → Start dose: 50% of standard (2.5–5 mg) [S8, S9]
#
# Case 2: Age 62, no hepatic impairment, citalopram
#   → Active paths: adult (age 62 < 65, no comorbidities)
#   → Cap: 20 mg from age > 60 rule [S1, S3] — independent of geriatric path
#   → Start dose: standard (10–20 mg)
# ============================================================

SEP = "─" * 64

def show_dose_detail(label, p, med_key):
    paths = detect_active_paths(p)
    cap   = max_cap_for_context(p, med_key, paths)
    std_start = MED_KB[med_key]["start_dose"]
    ctx_start = get_context_start_dose(p, med_key, paths)
    kb_max    = DOSE_MIN_MAX_MG.get(med_key, (None, None))[1]
    eff_max   = min(kb_max, cap) if (cap is not None and kb_max is not None) else kb_max

    print(f"\n{SEP}")
    print(f"  {label}")
    print(SEP)
    print(f"  Patient age       : {p.age}")
    print(f"  Hepatic impairment: {p.hepatic_impairment}")
    print(f"  Active paths      : {paths}")
    print(f"  Medication        : {MED_KB[med_key]['display']}")
    print(f"  KB standard max   : {kb_max} mg")
    print(f"  Context cap       : {cap} mg  {'← most restrictive across all paths' if cap else ''}")
    print(f"  Effective max     : {eff_max} mg")
    print(f"  Standard start    : {std_start}")
    print(f"  Context start     : {ctx_start or '(unchanged — use standard)'}")
    print()

    # Build a recommendation to show exactly what the clinician would see
    reco = _make_reco(p, med_key, "start", active_paths=paths)
    print(f"  Recommendation output (intent=start):")
    print(f"    display    : {reco.medication_display}")
    print(f"    start_dose : {reco.start_dose}")
    print(f"    dose_range : {reco.dose_range}")
    for msg in reco.messages:
        print(f"    note       : {msg}")


# ── Case 1: Age 68, hepatic impairment ───────────────────────────────────────
p1 = PatientInput(age=68, phq_current=16, hepatic_impairment=True)
show_dose_detail(
    "CASE 1 — Age 68, hepatic impairment | medication: escitalopram",
    p1, "escitalopram"
)
# Also show venlafaxine to confirm the 112.5mg hepatic cap fires
show_dose_detail(
    "CASE 1 (venlafaxine check) — Age 68, hepatic | medication: venlafaxine_xr",
    p1, "venlafaxine_xr"
)

# ── Case 2: Age 62, no hepatic impairment ────────────────────────────────────
p2 = PatientInput(age=62, phq_current=16)
show_dose_detail(
    "CASE 2 — Age 62, no hepatic impairment | medication: citalopram",
    p2, "citalopram"
)
# Confirm escitalopram is NOT capped for age 62 (age > 60 cap is citalopram-specific)
show_dose_detail(
    "CASE 2 (escitalopram sanity) — Age 62, no hepatic | medication: escitalopram",
    p2, "escitalopram"
)

# ── Summary assertions ────────────────────────────────────────────────────────
print(f"\n{'═'*64}")
print("  SUMMARY CHECKS")
print(f"{'═'*64}")

def cap_for(p, med): return max_cap_for_context(p, med, detect_active_paths(p))
def start_for(p, med): return get_context_start_dose(p, med, detect_active_paths(p))

checks = [
    # Duloxetine global max
    ("Duloxetine global max = 60 mg",
     DOSE_MIN_MAX_MG["duloxetine"][1] == 60),
    # Fluoxetine global max
    ("Fluoxetine global max = 80 mg",
     DOSE_MIN_MAX_MG["fluoxetine"][1] == 80),
    # Case 1: escitalopram cap = 10mg (geriatric + hepatic both cap at 10)
    ("Case 1: escitalopram cap = 10 mg",
     cap_for(p1, "escitalopram") == 10.0),
    # Case 1: escitalopram start dose reduced to 50%
    ("Case 1: escitalopram start dose is hepatic-reduced",
     start_for(p1, "escitalopram") is not None and "50%" in start_for(p1, "escitalopram")),
    # Case 1: venlafaxine cap = 112.5mg (hepatic 50% of 225)
    ("Case 1: venlafaxine_xr cap = 112.5 mg",
     cap_for(p1, "venlafaxine_xr") == 112.5),
    # Case 1: venlafaxine start dose reduced
    ("Case 1: venlafaxine_xr start dose is hepatic-reduced",
     start_for(p1, "venlafaxine_xr") is not None and "50%" in start_for(p1, "venlafaxine_xr")),
    # Case 2: citalopram cap = 20mg (age > 60, no geriatric path)
    ("Case 2: citalopram cap = 20 mg (age > 60 rule)",
     cap_for(p2, "citalopram") == 20.0),
    # Case 2: age 62 NOT on geriatric path
    ("Case 2: geriatric path NOT active (age 62)",
     "geriatric" not in detect_active_paths(p2)),
    # Case 2: escitalopram NOT capped for age 62 (only citalopram has the >60 rule)
    ("Case 2: escitalopram cap = None for age 62 (no age>60 rule for escitalopram)",
     cap_for(p2, "escitalopram") is None),
    # Case 2: citalopram start dose NOT reduced (no hepatic impairment)
    ("Case 2: citalopram start dose unchanged (no hepatic)",
     start_for(p2, "citalopram") is None),
]

all_passed = True
for label, passed in checks:
    print(f"  [{'PASS' if passed else 'FAIL'}]  {label}")
    if not passed: all_passed = False

print(f"\n  {'All checks passed.' if all_passed else 'FAILURES — review above.'}")


In [ ]:
import sys
sys.path.insert(0, '/tmp')
exec(open('/tmp/mdd_cell0.py').read())

SEP  = "─" * 68
SEP2 = "═" * 68

def show_aug(label, result):
    print(f"\n{SEP}")
    print(f"  {label}")
    print(SEP)
    print(f"  Pathways    : {result.pathway_applied}")
    branches = [a for a in result.audit_trail if a.startswith("branch:")]
    print(f"  Branch      : {branches}")
    print(f"  Stop reason : {result.stop_reason or '(none)'}")
    print()
    if result.non_med_recommendations:
        print("  Non-med recommendations:")
        for r in result.non_med_recommendations:
            print(f"    → {r}")
        print()
    if result.warnings:
        print("  Warnings:")
        for w in result.warnings:
            print(f"    ⚠  {w}")
        print()
    print("  Rationale:")
    for r in result.rationale:
        print(f"    • {r}")
    print()
    print("  Augmentation/recommendations:")
    for idx, r in enumerate(result.recommendations, 1):
        print(f"    [{idx}] {r.medication_display}  (intent={r.intent})")
        for rat in r.rationale:
            print(f"         Rationale: {rat}")
        if r.start_dose:
            print(f"         Start: {r.start_dose}")
        print(f"         Range: {r.dose_range}")
        for m in r.messages[:2]:
            print(f"         Note : {m}")

# ── CASE A: Trial 1, partial, currently on sertraline (SSRI) at max dose ─────
pA = PatientInput(
    age=42, phq_current=10, baseline_phq=18,
    weeks_on_current_antidepressant=8,
    current_antidepressant_key="sertraline",
    current_dose_mg=200,   # sertraline max = 200mg
    trial_number=1,
)
rA = engine.run(pA)
show_aug("CASE A — Trial 1, partial, sertraline SSRI at max dose", rA)

# ── CASE B: Trial 2, partial, good tolerability, on venlafaxine at max ───────
pB = PatientInput(
    age=45, phq_current=12, baseline_phq=20,
    weeks_on_current_antidepressant=10,
    current_antidepressant_key="venlafaxine_xr",
    current_dose_mg=225,   # venlafaxine max = 225mg
    trial_number=2,
    tolerability="good",
    prior_trials=[PriorTrial(medication_key="sertraline", outcome="no_response")],
)
rB = engine.run(pB)
show_aug("CASE B — Trial 2, partial, good tolerability, venlafaxine_xr at max", rB)

# ── CASE C: Trial 3, partial, on bupropion at max ────────────────────────────
pC = PatientInput(
    age=50, phq_current=14, baseline_phq=22,
    weeks_on_current_antidepressant=10,
    current_antidepressant_key="bupropion_xl",
    current_dose_mg=450,   # bupropion_xl max = 450mg
    trial_number=3,
    tolerability="good",
    prior_trials=[
        PriorTrial(medication_key="sertraline", outcome="no_response"),
        PriorTrial(medication_key="venlafaxine_xr", outcome="no_response"),
    ],
)
rC = engine.run(pC)
show_aug("CASE C — Trial 3, partial, bupropion at max dose", rC)

# ── Summary assertions ────────────────────────────────────────────────────────
print(f"\n{SEP2}")
print("  SUMMARY CHECKS")
print(SEP2)

def first_aug_med(result):
    return next((r.medication_key for r in result.recommendations if r.intent == "augment_with"), None)

def has_aug_tier(result, n):
    tier_str = f"{n} \u2014"
    return any(
        any(tier_str in e.value for e in r.evidence)
        for r in result.recommendations
        if r.intent == "augment_with"
    )

def has_branch(result, b):
    return any(b in a for a in result.audit_trail)

checks = [
    # Case A: Round 5 Tier 1 = aripiprazole (always) + bupropion (if on SSRI/SNRI)
    # Sertraline is SSRI → both aripiprazole and bupropion are Tier 1; aripiprazole added first
    ("A: first augmenter is aripiprazole (Round 5 Tier 1)",
     first_aug_med(rA) == "aripiprazole"),
    ("A: tier-1 augmenter evidence present",
     has_aug_tier(rA, 1)),
    ("A: tier-2 SGA also present",
     has_aug_tier(rA, 2)),
    ("A: tier-3 lithium present",
     has_aug_tier(rA, 3)),
    ("A: branch=step_6_trial1_augment",
     has_branch(rA, "step_6_trial1")),
    ("A: bupropion also offered as Tier 1 (SSRI patient)",
     any(r.medication_key == "bupropion_xl" and r.intent == "augment_with"
         for r in rA.recommendations)),

    # Case B: step 6a fires, augmentation offered before trial 3
    ("B: branch=step_6a_good_tolerability",
     has_branch(rB, "step_6a_good_tolerability")),
    ("B: first augmenter is aripiprazole (Round 5 Tier 1)",
     first_aug_med(rB) == "aripiprazole"),
    ("B: bupropion also offered as Tier 1 (SNRI patient)",
     any(r.medication_key == "bupropion_xl" and r.intent == "augment_with"
         for r in rB.recommendations)),
    ("B: no switch intent (augmentation preferred)",
     not any(r.intent == "switch_to" for r in rB.recommendations)),

    # Case C: step 6b fires; bupropion is current med → NOT offered as augmenter
    ("C: branch=step_6b_trial3_partial",
     has_branch(rC, "step_6b_trial3_partial")),
    ("C: psychiatric consultation in non_med_recommendations",
     any("Psychiatric consultation" in r for r in rC.non_med_recommendations)),
    ("C: advanced interventions flagged (ECT mentioned)",
     any("ECT" in r for r in rC.non_med_recommendations)),
    ("C: STAR*D note in warnings",
     any("STAR*D" in w for w in rC.warnings)),
    # Case C: bupropion is current med → aripiprazole is first Tier 1 augmenter
    ("C: first augmenter is aripiprazole (bupropion already current)",
     first_aug_med(rC) == "aripiprazole"),
]

all_ok = True
for label, passed in checks:
    print(f"  [{'PASS' if passed else 'FAIL'}]  {label}")
    if not passed:
        all_ok = False

print(f"\n  {'All checks passed.' if all_ok else 'FAILURES — review above.'}")


In [ ]:
# ============================================================
# Test: Round 5 — five output sections + four-tier augmentation
#
# Patient: Age 45, PHQ-9 17, first episode, Trial 1, adult path
#          sertraline 50 mg (starting dose), suicidality: none
#
# Expected:
#   1. Recommendations     → escitalopram + sertraline (standard first-line)
#   2. Monitoring schedule → 5 timepoints (Week 2, 4–6, 8–12, initiation, sui weeks 1–4)
#   3. Maintenance plan    → first-episode: 6–9 months after remission [2, 4, 71]
#   4. Medications excluded → none (clean adult path, no comorbidities)
#   5. Adjunctive options  → omega-3 (SOR B) + celecoxib (no contraindications)
#   6. Reference list      → auto-collected from all cited [n] tags in output
# ============================================================

SEP  = "─" * 70
SEP2 = "═" * 70

p = PatientInput(
    age=45,
    phq_current=17,
    prior_depressive_episodes=0,      # first episode → 6–9 month maintenance
    current_antidepressant_key="sertraline",
    current_dose_mg=50,               # starting dose
    trial_number=1,
    suicidality="none",
    tolerability="good",
)
r = engine.run(p)

print(f"\n{SEP2}")
print("  ROUND 5 TEST — Age 45, PHQ 17, first episode, trial 1, sertraline 50 mg")
print(SEP2)
print(f"\n  Pathways    : {r.pathway_applied}")
print(f"  Stop reason : {r.stop_reason or '(none)'}")

print(f"\n{SEP}")
print("  1. RECOMMENDATIONS")
print(SEP)
for i, rec in enumerate(r.recommendations, 1):
    print(f"  [{i}] {rec.medication_display}  (intent={rec.intent})")
    if rec.start_dose:
        print(f"       Start  : {rec.start_dose}")
    print(f"       Range  : {rec.dose_range}")
    for rat in rec.rationale:
        print(f"       Reason : {rat}")

print(f"\n{SEP}")
print("  2. MONITORING SCHEDULE")
print(SEP)
for line in r.monitoring_schedule:
    print(f"  • {line}")

print(f"\n{SEP}")
print("  3. MAINTENANCE PLAN")
print(SEP)
for line in r.maintenance_plan:
    print(f"  • {line}")

print(f"\n{SEP}")
print("  4. MEDICATIONS EXCLUDED")
print(SEP)
for line in r.medications_excluded:
    print(f"  • {line}")

print(f"\n{SEP}")
print("  5. ADJUNCTIVE OPTIONS (TIER 4)")
print(SEP)
for line in r.adjunctive_options:
    print(f"  • {line}")

print(f"\n{SEP}")
print("  6. REFERENCE LIST")
print(SEP)
for ref in r.reference_list:
    print(f"  {ref}")

# ── Summary assertions ────────────────────────────────────────────────────────
print(f"\n{SEP2}")
print("  SUMMARY CHECKS")
print(SEP2)

checks = [
    ("monitoring_schedule is populated",
     len(r.monitoring_schedule) >= 5),
    ("maintenance_plan is populated",
     len(r.maintenance_plan) >= 1),
    ("medications_excluded is populated",
     len(r.medications_excluded) >= 1),
    ("adjunctive_options is populated",
     len(r.adjunctive_options) >= 2),
    ("reference_list is populated",
     len(r.reference_list) >= 5),
    ("maintenance: first episode → 6–9 months",
     any("6" in m and "9" in m for m in r.maintenance_plan)),
    ("monitoring: Week 2 entry present",
     any("Week 2" in m for m in r.monitoring_schedule)),
    ("monitoring: Week 8–12 entry present",
     any("8" in m and "12" in m for m in r.monitoring_schedule)),
    ("adjunctive: omega-3 present",
     any("Omega-3" in a for a in r.adjunctive_options)),
    ("adjunctive: celecoxib present",
     any("Celecoxib" in a or "celecoxib" in a for a in r.adjunctive_options)),
    ("celecoxib NOT contraindicated (no cardiac/renal)",
     not any("CONTRAINDICATED" in a for a in r.adjunctive_options)),
    ("reference list includes 4 (VA/DoD)",
     any(ref.startswith("4.") for ref in r.reference_list)),
    ("reference list includes 26 (JAMA 2024)",
     any(ref.startswith("26.") for ref in r.reference_list)),
    ("no medications blocked (clean adult path)",
     any("No medications excluded" in m for m in r.medications_excluded)),
    ("recommendations not empty",
     len(r.recommendations) > 0),
    ("first recommendation is start intent",
     r.recommendations[0].intent == "start"),
]

all_ok = True
for label, passed in checks:
    print(f"  [{'PASS' if passed else 'FAIL'}]  {label}")
    if not passed:
        all_ok = False

print(f"\n  {'All checks passed.' if all_ok else 'FAILURES — review above.'}")


In [ ]:
import sys, json
exec(open('/tmp/mdd_cell0.py').read())

SEP  = "─" * 74
SEP2 = "═" * 74

def show_case(label, result):
    print(f"\n{SEP2}")
    print(f"  {label}")
    print(SEP2)
    if result.text_report:
        print(result.text_report)
    else:
        print("  (no text_report generated)")
    print(f"\n  audit: {result.audit_trail[:8]}")

# ══════════════════════════════════════════════════════════════════════════
# Case A — PHQ 7, mild, first episode, trial 1
# Expected: medications absent, therapy MoodCalmer self-directed,
#           no sections 2/3/5/6/7, next check-in Week 8-12
# ══════════════════════════════════════════════════════════════════════════
pA = PatientInput(
    age=38,
    phq_current=7,
    prior_depressive_episodes=0,
    trial_number=1,
    suicidality="none",
)
rA = engine.run(pA)
show_case("CASE A — PHQ 7 mild, first episode, trial 1", rA)

# ══════════════════════════════════════════════════════════════════════════
# Case B — PHQ 12, moderate, first episode, trial 1
# Expected: medications present (shared decision), therapy both options,
#           no sections 2/3/5/6/7, next check-in Week 2 (new start)
# ══════════════════════════════════════════════════════════════════════════
pB = PatientInput(
    age=45,
    phq_current=12,
    prior_depressive_episodes=0,
    trial_number=1,
    suicidality="none",
)
rB = engine.run(pB)
show_case("CASE B — PHQ 12 moderate, first episode, trial 1", rB)

# ══════════════════════════════════════════════════════════════════════════
# Case C — PHQ 17, moderate-severe, trial 2, venlafaxine 150mg, 8 weeks,
#           no response (18% PHQ reduction < 25%)
# Expected: switching protocol slow taper, face-to-face therapy,
#           safety flags for extended taper, next check-in Week 2
# ══════════════════════════════════════════════════════════════════════════
pC = PatientInput(
    age=45,
    phq_current=17,
    baseline_phq=22,
    prior_depressive_episodes=0,
    trial_number=2,
    current_antidepressant_key="venlafaxine_xr",
    current_dose_mg=150,
    weeks_on_current_antidepressant=8,
    suicidality="none",
    tolerability="good",
    prior_trials=[PriorTrial(medication_key="sertraline", outcome="no_response")],
)
rC = engine.run(pC)
show_case("CASE C — PHQ 17 mod-severe, trial 2, venlafaxine → switch", rC)

# ══════════════════════════════════════════════════════════════════════════
# SUMMARY CHECKS
# ══════════════════════════════════════════════════════════════════════════
print(f"\n{SEP2}")
print("  SUMMARY CHECKS")
print(SEP2)

def report_has(result, section):
    return result.report is not None and section in result.report

def report_section(result, *keys):
    r = result.report
    for k in keys:
        if r is None: return None
        r = r.get(k)
    return r

def text_has(result, text):
    return result.text_report is not None and text.lower() in result.text_report.lower()

checks = [
    # ── Case A ─────────────────────────────────────────────────────────────
    ("A: report populated",
     rA.report is not None),
    ("A: text_report populated",
     rA.text_report is not None and len(rA.text_report) > 50),
    ("A: patient_summary has phq=7",
     report_section(rA, "patient_summary", "phq_score") == 7),
    ("A: severity is mild",
     report_section(rA, "patient_summary", "severity") == "Mild (PHQ 5\u20139)"),
    ("A: no medications section (PHQ 7, first episode)",
     report_section(rA, "recommendations", "medications") is None
     or len(report_section(rA, "recommendations", "medications") or []) == 0),
    ("A: therapy present (mild → MoodCalmer self-directed)",
     report_section(rA, "recommendations", "therapy") is not None),
    ("A: therapy level is self-directed digital CBT",
     (report_section(rA, "recommendations", "therapy") or {}).get("level") == "self-directed digital CBT"),
    ("A: no safety_flags section",
     not report_has(rA, "safety_flags")),
    ("A: no switching_protocol section",
     not report_has(rA, "switching_protocol")),
    ("A: no augmentation_plan section",
     not report_has(rA, "augmentation_plan")),
    ("A: next_check_in present",
     report_section(rA, "next_check_in") is not None),
    ("A: next_check_in Week 8–12",
     "8" in (report_section(rA, "next_check_in") or "") and "12" in (report_section(rA, "next_check_in") or "")),
    ("A: references include 99 or 101 (CBT refs)",
     "99" in (report_section(rA, "references") or {}) or "101" in (report_section(rA, "references") or {})),
    ("A: text_report mentions MoodCalmer",
     text_has(rA, "moodcalmer")),

    # ── Case B ─────────────────────────────────────────────────────────────
    ("B: report populated",
     rB.report is not None),
    ("B: severity is moderate",
     report_section(rB, "patient_summary", "severity") == "Moderate (PHQ 10\u201314)"),
    ("B: medications present (PHQ 12 → recommend)",
     len(report_section(rB, "recommendations", "medications") or []) > 0),
    ("B: therapy present (PHQ 12 → shared decision)",
     report_section(rB, "recommendations", "therapy") is not None),
    ("B: therapy level is shared decision",
     (report_section(rB, "recommendations", "therapy") or {}).get("level") == "shared decision"),
    ("B: no switching_protocol section",
     not report_has(rB, "switching_protocol")),
    ("B: no augmentation_plan section",
     not report_has(rB, "augmentation_plan")),
    ("B: next_check_in Week 2 (new start)",
     "week 2" in (report_section(rB, "next_check_in") or "").lower()),
    ("B: medication doses include Start dose",
     any("Start:" in (m.get("dose") or "") for m in (report_section(rB, "recommendations", "medications") or []))),
    ("B: references populated",
     len(report_section(rB, "references") or {}) > 0),
    ("B: text_report mentions shared decision",
     text_has(rB, "shared decision")),

    # ── Case C ─────────────────────────────────────────────────────────────
    ("C: report populated",
     rC.report is not None),
    ("C: severity is moderate-severe",
     report_section(rC, "patient_summary", "severity") == "Moderate-Severe (PHQ 15\u201319)"),
    ("C: phq_improvement_pct present (18%)",
     report_section(rC, "patient_summary", "phq_improvement_pct") is not None),
    ("C: switching_protocol section present",
     report_has(rC, "switching_protocol") and len(report_section(rC, "switching_protocol")) > 0),
    ("C: switching protocol method is Slow taper",
     any("Slow taper" in (e.get("method") or "") for e in (report_section(rC, "switching_protocol") or []))),
    ("C: switching protocol warning present",
     any(e.get("warning") for e in (report_section(rC, "switching_protocol") or []))),
    ("C: therapy present (PHQ 17 → recommended combination)",
     report_section(rC, "recommendations", "therapy") is not None),
    ("C: therapy level is recommended combination",
     (report_section(rC, "recommendations", "therapy") or {}).get("level") == "recommended combination"),
    ("C: therapy format is face-to-face",
     "face-to-face" in (report_section(rC, "recommendations", "therapy") or {}).get("format", "")),
    ("C: medications present (switch_to recs)",
     len(report_section(rC, "recommendations", "medications") or []) > 0),
    ("C: switch_to medications have Target dose (not Start)",
     any(
         "Target:" in (m.get("dose") or "") and "Start:" not in (m.get("dose") or "")
         for m in (report_section(rC, "recommendations", "medications") or [])
         if m.get("intent") == "switch_to"
     )),
    ("C: next_check_in Week 2 (switch)",
     "week 2" in (report_section(rC, "next_check_in") or "").lower()),
    ("C: text_report mentions Slow taper",
     text_has(rC, "slow taper")),
    ("C: references include switching refs (87 or 93)",
     "87" in (report_section(rC, "references") or {}) or "93" in (report_section(rC, "references") or {})),
    ("C: valid JSON (report dict)",
     json.dumps(rC.report) is not None),
]

all_ok = True
for label, passed in checks:
    print(f"  [{'PASS' if passed else 'FAIL'}]  {label}")
    if not passed:
        all_ok = False

print(f"\n  {'All checks passed.' if all_ok else 'FAILURES — review above.'}")

# Bonus: validate Case C JSON round-trip
print(f"\n{SEP}")
print("  JSON VALIDATION (Case C)")
print(SEP)
try:
    j = json.dumps(rC.report, default=str, indent=2)
    parsed = json.loads(j)
    print(f"  JSON round-trip OK — {len(j):,} chars, {len(parsed)} top-level keys")
    print(f"  Keys: {list(parsed.keys())}")
except Exception as e:
    print(f"  JSON ERROR: {e}")
